# Ранжирование на основе datamart

В этом ноутбуке построена приближенная к реальной рекомендательная система. Работа првоедена с данными marketplace из [T-ECD](https://huggingface.co/datasets/t-tech/T-ECD).

Обычно рекомендательная система состоит из нескольких этапов:
1. Отбор кандидатов (Retrieval)
2. Ранжирование (Ranking)
3. Бизнес-логика (например, условие на то, чтобы товары от одного продавца не стояли в ленте друг за другом)

В этой работе сосредоточимся на втором этапе.

Краткое напоминание, почему отбор кандидатов и ранжирование - разные этапы. Задачу рекомендаций можно решать как регрессию (насколько релевантен айтем), классификацию (релевантен ли айтем) или ранжирование (какой из двух айтемов релевантнее). В идеале - проранжировать каталог под каждого пользователя. Но каталог всегда существенно больше того подмножества айтемов, которые пользователь увидит в итоговой выдаче. А качественно ранжировать весь каталог - ОЧЕНЬ долго и дорого. Получаем trade-off скорости и качества. Простое решение - многостадийные рекомендации. Сначала отберем кандидатов (релевантные/не релевантные), а потом проранжируем только релевантные.

In [1]:
pip install -q polars lightgbm scikit-learn catboost shap optuna implicit torch

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import json
import gc
import os
import random
import typing as t
from abc import ABC, abstractmethod
from collections import defaultdict
from dataclasses import dataclass
from functools import partial
from pathlib import Path

import joblib
import lightgbm as lgbm
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import polars as pl
import polars.selectors as cs
import shap
import torch
import torch.nn as nn
import torch.optim as optim
from IPython.display import HTML
from implicit.als import AlternatingLeastSquares
from optuna.samplers import TPESampler
from scipy.sparse import coo_matrix, csr_matrix
from tqdm import tqdm
from torch.utils.data import DataLoader, Dataset, IterableDataset
import math

## Download Data

Данные занимают около 3.5 GB

In [ ]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id='t-tech/T-ECD',
    repo_type='dataset',
    local_dir='.',
    local_dir_use_symlinks=False,
    allow_patterns=['dataset/small/users.pq', 'dataset/small/marketplace/**']
)

c:\Users\Mi\Desktop\study\recsys_course\hw3\.venv\Lib\site-packages\huggingface_hub\utils\_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
Fetching ... files: 0it [00:00, ?it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
Fetching ... files: 229it [02:10,  1.75it/s]


'C:\\Users\\Mi\\Desktop\\study\\recsys_course\\hw3'

Больше всего места занимают эмбеддинги айтемов, я их удалю, так как в этой работе нам потребуется дополнительное место на создание датамарта

In [ ]:
# pl.read_parquet('dataset/small/marketplace/items.pq').drop('embedding').write_parquet('dataset/small/marketplace/items.pq')

''''
на локальной машине код упал с ошибкой 'OSError: Запрошенную операцию нельзя выполнить...' поэтому эмбеддинги
пришлось удалить из items.pq через пандас
'''

src = 'dataset/small/marketplace/items.pq'
tmp = 'dataset/small/marketplace/items_no_embedding.pq'

cols = [c for c in pl.read_parquet_schema(src) if c != 'embedding']

pl.scan_parquet(src).select(cols).sink_parquet(tmp)

gc.collect()
os.replace(tmp, src)

## I. Datamart (4 балла)

В задаче ранжирования я буду ориентироваться на бустинги ([Catboost](https://catboost.ai), [LGBM](https://lightgbm.readthedocs.io/en/latest/pythonapi/lightgbm.Booster.html), [XGBoost](https://xgboost.readthedocs.io/en/stable/)), потому что они хорошо себя показывают. С точки зрения интерфейса буду использовать: Algorithm(user, item, [features]), где фичи - любые полезные статистики (количество просмотров, конверсия из клика в кликаут, средний рейтинг, ...).

Каждый раз рассчитывать фичи с нуля по сырым логам достаточно тяжело (а в реальности рекомендации мы делаем не один раз в домашней работе, а гораздо чаще). Учитывая, что логи могут иметь разный формат или нуждаться в  дополнительной фильтрации (баги в логах всё же не редкость), я предподсчитаю статистики и сохраню их в отдельных файлах, которые затем удобно просто прочитать. 

Это удобно сделать посредством датамарта. Датамарт - это витрина данных под определенную задачу. Он состоит из слоёв, где переход между слоями задает преобразование над данными, и обычно такие преобразования выполняются раз в какое-то время (в моем случае пусть будет день). Для задачи ранжирования мне потребуется три слоя: 
1. Raw - содержит сырые логи за каждый день. Я уже собрал его на предыдущем шаге.
2. Aggs - будет содержать агрегированные статистики по юзерам и айтемам за каждый день (user, item, [stats]). Например, количество просмотров, количество кликов, количество кликов c поверхности поиска.
3. Features - будет содержать фичи, которые я хочу использовать в модели, тоже за каждый день. Например, средний рейтинг айтема, средний рейтинг айтема по категориям, общая конверсия из клика в кликаут для пользователя за последние 30 дней. На этом слое удобно выделить отдельные папки по группам фичей (user, item, user-item, ...), чтобы избежать дубликатов при хранении. 

Возьмем только небольшой срез данных, иначе дальнейшая работа может стать computationally infeasible. Переложим этот срез в `datamart/raw/events/{action_type}/{day}.pq`

Именно в таком формате логи обычно хранятся в сервисе. 

Вы можете расширить условия на сэмплирование юзеров и айтемов. Если у вас будут проблемы с памятью, то можете и уменьшить что-то, но чем меньше ваш датасет, тем хуже будут метрики у конечной модели

In [3]:
ACTION_TYPES = ['view', 'click', 'clickout', 'like']
SUBDOMAINS = ['u2i', 'i2i', 'catalog', 'search', 'other']

DAYS = list(range(1250, 1301))  # не все даты [как я понял мы дни 1301-1308 отложим для валидации]

Будем считать, что view < click < clickout < like с точки зрения бизнеса. Этот факт будет использоваться далее.

In [4]:
selected_users = (
    pl.concat(
        [pl.scan_parquet(f'dataset/small/marketplace/events/{str(day).zfill(5)}.pq') for day in DAYS]
    )
    .group_by('user_id').agg(pl.len()).sort('len', descending=True)
    .head(20000).collect()['user_id'].to_list()
)

selected_items = (
    pl.concat(
        [pl.scan_parquet(f'dataset/small/marketplace/events/{str(day).zfill(5)}.pq') for day in DAYS]
    ).group_by('item_id').agg(pl.len()).sort('len', descending=True)
    .head(20000).collect()['item_id'].to_list()
)

In [5]:
display(pd.DataFrame(selected_users[:10]))
display(pd.DataFrame(selected_items[:10]))

,0
0,35768329
1,14968784
2,61159143
3,54147498
4,13112529
5,5181034
6,84103532
7,7112029
8,56540287
9,20306353


,0
0,nfmcg_18106422
1,nfmcg_19353812
2,nfmcg_8961427
3,nfmcg_13647154
4,nfmcg_8797298
5,nfmcg_22204691
6,nfmcg_22211380
7,nfmcg_27366316
8,nfmcg_9041331
9,nfmcg_4410609


In [5]:
USERS = pl.scan_parquet('dataset/small/users.pq').filter(pl.col('user_id').is_in(selected_users)).collect()
print(USERS.shape)
ITEMS = pl.scan_parquet('dataset/small/marketplace/items.pq').filter(pl.col('item_id').is_in(selected_items)).collect()
print(ITEMS.shape)


(20000, 3)
(20000, 5)


In [7]:
display(USERS.head())
display(ITEMS.head())

user_id,socdem_cluster,region
u64,u8,u8
20837154,9,58
37335450,20,67
79718992,17,13
39909766,19,24
10670013,0,41


item_id,brand_id,category,subcategory,price
str,u64,str,str,f64
"""nfmcg_10000326""",117885,"""Electronic Devices and Gadgets""","""Mobile Devices and Electronic …",4.270476
"""nfmcg_10000983""",227580,null,null,-1.363543
"""nfmcg_10001681""",137356,null,null,3.931349
"""nfmcg_10003516""",28667,"""Electronic Devices and Gadgets""","""Mobile Devices and Electronic …",4.896499
"""nfmcg_10004693""",189615,"""Fashion Accessories, Tech Add-…","""Accessories for Storing Small …",-3.666876


### I.I. Datamart -> Raw слой (1 из 4 баллов)

Здесь будет собран raw слой:

![](images/datamart_raw.png)

в каждом файлике должны храниться данные на конкретную дату


In [8]:
# срез в `datamart/raw/events/{action_type}/{day}.pq`

# SUBDOMAINS = ['u2i', 'i2i', 'catalog', 'search', 'other'] <- не забыть сделать по субдоменам

raw_events_dir = Path('datamart/raw/events/')

for action_type in ACTION_TYPES: # ['view', 'click', 'clickout', 'like']
    (raw_events_dir / action_type).mkdir(parents=True, exist_ok=True)


for day in tqdm(DAYS):
    day_name = str(day).zfill(5)
    src_path = Path(f'dataset/small/marketplace/events/{day_name}.pq')
    
    day_events = (
        pl.scan_parquet(src_path)
        .filter(
            pl.col('user_id').is_in(selected_users)
            & pl.col('item_id').is_in(selected_items)
        )
        .collect()
    )
    
    for action_type in ACTION_TYPES:
        (
            day_events
            .filter(pl.col('action_type') == action_type)
            .write_parquet(raw_events_dir / action_type / f'{day_name}.pq')
        )


  0%|          | 0/51 [00:00<?, ?it/s]

100%|██████████| 51/51 [00:02<00:00, 23.79it/s]


In [8]:
# `datamart/raw/subdomains/{subdomain}/{day}.pq`

raw_events_dir = Path('datamart/raw/subdomains/')
for subdomain in SUBDOMAINS: # ['u2i', 'i2i', 'catalog', 'search', 'other']
    (raw_events_dir / subdomain).mkdir(parents=True, exist_ok=True)


for day in tqdm(DAYS):
    day_name = str(day).zfill(5)
    src_path = Path(f'dataset/small/marketplace/events/{day_name}.pq')
    
    day_events = (
        pl.scan_parquet(src_path)
        .filter(
            pl.col('user_id').is_in(selected_users)
            & pl.col('item_id').is_in(selected_items)
        )
        .collect()
    )
    
    for subdomain in SUBDOMAINS:
        (
            day_events
            .filter(pl.col('subdomain') == subdomain)
            .write_parquet(raw_events_dir / subdomain / f'{day_name}.pq')
        )


100%|██████████| 51/51 [00:02<00:00, 23.22it/s]


### I.II. Datamart -> Agg слой (1 из 4 баллов)

Я рассчитаю количество событий каждого типа (`action_type`) по каждой поверхности (`subdomain`) для пар (`user_id`, `item_id`) за каждый день. Также добавлю общий счетчик — сумму по всем поверхностям. Результат сохраню в виде polars-таблиц.

Формат будет аналогичен raw, но в agg значения внутри дня уже будут агрегированы.

![](images/datamart_agg.png)

In [10]:
# `datamart/aggs/events/{day}.pq`

# act_type - ['view', 'click', 'clickout', 'like']
# subd - ['catalog', 'i2i', 'u2i', 'search', 'other']

agg_events_dir = Path('datamart/aggs/events')
agg_events_dir.mkdir(parents=True, exist_ok=True)

count_columns = [f'num_{action_type}_all_subdomains' for action_type in ACTION_TYPES] + \
[f'num_{action_type}_{subdomain}' for action_type in ACTION_TYPES for subdomain in SUBDOMAINS] # 

for day in tqdm(DAYS):
    day_name = str(day).zfill(5)
    src_path = Path(f'dataset/small/marketplace/events/{day_name}.pq')
    
    day_events = (
        pl.scan_parquet(src_path)
        .filter(
            pl.col('user_id').is_in(selected_users)
            & pl.col('item_id').is_in(selected_items)
        )
    )
    
    day_aggs = (
        day_events
        .group_by('user_id', 'item_id')
        .agg(
            [
                (pl.col('action_type') == action_type)
                .sum()
                .cast(pl.UInt16)
                .alias(f'num_{action_type}_all_subdomains')
                for action_type in ACTION_TYPES
            ]
            + [
                (
                    (pl.col('action_type') == action_type)
                    & (pl.col('subdomain') == subdomain)
                )
                .sum()
                .cast(pl.UInt16)
                .alias(f'num_{action_type}_{subdomain}')
                for action_type in ACTION_TYPES
                for subdomain in SUBDOMAINS
            ]
        )
        .select(['user_id', 'item_id'] + count_columns)
        .collect()
    )
    
    day_aggs.write_parquet(agg_events_dir / f'{day_name}.pq')

100%|██████████| 51/51 [00:01<00:00, 29.74it/s]


In [ ]:
pl.read_parquet('datamart/aggs/events/01300.pq').sample(10) 

user_id,item_id,agg_num_view_all_subdomains,agg_num_click_all_subdomains,agg_num_clickout_all_subdomains,agg_num_like_all_subdomains,agg_num_view_u2i,agg_num_view_i2i,agg_num_view_catalog,agg_num_view_search,agg_num_view_other,agg_num_click_u2i,agg_num_click_i2i,agg_num_click_catalog,agg_num_click_search,agg_num_click_other,agg_num_clickout_u2i,agg_num_clickout_i2i,agg_num_clickout_catalog,agg_num_clickout_search,agg_num_clickout_other,agg_num_like_u2i,agg_num_like_i2i,agg_num_like_catalog,agg_num_like_search,agg_num_like_other
u64,str,u16,u16,u16,u16,u16,u16,u16,u16,u16,u16,u16,u16,u16,u16,u16,u16,u16,u16,u16,u16,u16,u16,u16,u16
55629343,"""nfmcg_15328188""",1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
15168114,"""nfmcg_27159615""",2,0,0,0,0,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
9342066,"""nfmcg_19329605""",1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
44769997,"""nfmcg_14929282""",0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0
33529676,"""nfmcg_23389912""",1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
29653656,"""nfmcg_14733850""",1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
13245611,"""nfmcg_9082123""",1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
46583354,"""nfmcg_27400983""",2,0,0,0,0,0,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
83344586,"""nfmcg_22967103""",1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


### I.III. Datamart -> Feature слой (2 из 4 баллов)

Я продумаю, какие признаки на основе уже собранных статистик можно использовать как фичи для модели, и реализую их подсчет за каждый день. Все признаки буду считать по временному окну, например: “количество кликов на товар за последние 14 дней”.

Сохраню признаки в виде polars-таблиц и разложу их по отдельным папкам по сущностям, чтобы не хранить дубликаты.

Минимально соберу 10 признаков:

* 2 признака только для сущности “пользователь”, например медианная цена кликнутых айтемов у этого пользователя
* 2 признака только для сущности “айтем”, например средняя конверсия из клика в кликаут по поверхности “поиск” у этого айтема
* 6 признаков для связи пользователя и айтема, например количество просмотров этого айтема у этого пользователя

Также постараюсь собрать больше признаков, потому что в данных есть разные сущности: категория, соцдем-кластер, бренд, а также параметры вроде цены, поверхности и типа события. Дополнительно можно посчитать изменение признака относительно предыдущего дня.

Для шаблонной логики буду использовать polars expressions, чтобы удобно подставлять нужные сущности и параметры и получать готовый набор признаков. Цену товаров буду считать фиксированной и брать из каталога `ITEMS`.

Также может быть полезно посчитать конверсии как отношение числа второго события из пары к числу первого события из пары. При этом я отдельно учту деление на ноль: пользователей, для которых число первого события из пары не определено или равно нулю, скорее всего не буду учитывать при таких расчетах.


![](images/datamart_feat2.png)

In [10]:
CONVERSION_PAIRS = [
    ('view', 'click'), 
    ('view', 'clickout'), 
    ('view', 'like'),
    ('click', 'clickout'),
    ('click', 'like'),
    ('clickout', 'like'),
]

Для удобства наименования фичей

In [11]:
def _feature_name(
    feature: str,
    keys: list[str],
    type: t.Literal['num', 'cat'] = 'num',
) -> str:
    return f"f_{type}__{'_'.join(keys)}__{feature}"


In [12]:
def _existing_agg_days(day_from: int, day_to: int, agg_dir: Path) -> list[int]:
    return sorted(int(path.stem) for path in agg_dir.glob('*.pq') if path.stem.isdigit() and day_from <= int(path.stem) <= day_to)


def _safe_ratio(num: pl.Expr, den: pl.Expr) -> pl.Expr:
    return pl.when(den > 0).then(num / den).otherwise(None).cast(pl.Float32)


def _write_or_merge_features(result: pl.DataFrame, output_path: Path, keys: list[str]) -> None:
    if output_path.exists():
        existing = pl.read_parquet(output_path)
        feature_cols_to_replace = [
            col for col in result.columns
            if col not in keys and col in existing.columns
        ]
        if feature_cols_to_replace:
            existing = existing.drop(feature_cols_to_replace)
        result = existing.join(result, on=keys, how='full', coalesce=True)

    result.write_parquet(output_path)


def calculate_user_group_item_group_features(
    day_from: int = 1200,
    day_to: int = 1300,
    num_days: int = 14,
    user_group: t.Literal['region', 'socdem_cluster'] | None = None,
    item_group: t.Literal['brand_id', 'category', 'subcategory', 'item_id'] = 'item_id'
) -> None:
    global USERS, ITEMS

    agg_dir = Path('datamart/aggs/events')
    output_root = Path('datamart/features/events')
    days = _existing_agg_days(day_from, day_to, agg_dir)
    if not days:
        return

    keys = ([user_group] if user_group is not None else []) + [item_group]
    output_dir = output_root / '-'.join(keys)
    output_dir.mkdir(parents=True, exist_ok=True)

    count_columns = {
        ('all_subdomains', action_type): f'num_{action_type}_all_subdomains'
        for action_type in ACTION_TYPES
    } | {
        (subdomain, action_type): f'num_{action_type}_{subdomain}'
        for action_type in ACTION_TYPES
        for subdomain in SUBDOMAINS
    }
    surfaces = ['all_subdomains'] + SUBDOMAINS

    for day in tqdm(days):
        window_days = [d for d in days if day - num_days < d <= day]
        if not window_days:
            continue

        events = pl.concat(
            [pl.scan_parquet(agg_dir / f'{d:05d}.pq') for d in window_days],
            how='vertical',
        )
        rename_map = {
            f'agg_num_{action_type}_all_subdomains': f'num_{action_type}_all_subdomains'
            for action_type in ACTION_TYPES
        } | {
            f'agg_num_{action_type}_{subdomain}': f'num_{action_type}_{subdomain}'
            for action_type in ACTION_TYPES
            for subdomain in SUBDOMAINS
        }
        source_columns = set(events.collect_schema().names())
        events = events.rename({old: new for old, new in rename_map.items() if old in source_columns})

        events = events.join(ITEMS.lazy(), on='item_id', how='left')
        if user_group is not None:
            events = events.join(USERS.lazy(), on='user_id', how='left')

        count_aggs = [
            pl.col(column).sum().cast(pl.Float32).alias(f'__num_{action_type}_{surface}')
            for surface in surfaces
            for action_type, column in [(action_type, count_columns[(surface, action_type)]) for action_type in ACTION_TYPES]
        ]
        median_aggs = [
            pl.when(pl.col(column) > 0)
            .then(pl.col('price'))
            .otherwise(None)
            .median()
            .cast(pl.Float32)
            .alias(_feature_name(f"median_price_{action_type}{'' if surface == 'all_subdomains' else f'_from_{surface}'}_{num_days}d", keys))
            for surface in surfaces
            for action_type, column in [(action_type, count_columns[(surface, action_type)]) for action_type in ACTION_TYPES]
        ]

        grouped = events.group_by(keys).agg(count_aggs + median_aggs)

        conv_exprs = [
            _safe_ratio(
                pl.col(f'__num_{target_action}_{surface}'),
                pl.col(f'__num_{source_action}_{surface}'),
            ).alias(_feature_name(
                f"conv_from_{source_action}_to_{target_action}{'' if surface == 'all_subdomains' else f'_from_{surface}'}_{num_days}d",
                keys,
            ))
            for source_action, target_action in CONVERSION_PAIRS
            for surface in surfaces
        ]
        share_exprs = [
            _safe_ratio(
                pl.col(f'__num_{action_type}_{surface}'),
                pl.col(f'__num_{action_type}_{surface}').sum(),
            ).alias(_feature_name(
                f"share_{action_type}{'' if surface == 'all_subdomains' else f'_from_{surface}'}_{num_days}d",
                keys,
            ))
            for action_type in ACTION_TYPES
            for surface in surfaces
        ]

        result = (
            grouped
            .with_columns(conv_exprs + share_exprs)
            .select(keys + [cs.starts_with('f_num__')])
            .collect()
        )
        _write_or_merge_features(result, output_dir / f'{day:05d}.pq', keys)


def calculate_user_item_group_features(
    day_from: int = 1200,
    day_to: int = 1300,
    num_days: int = 14,
    item_group: t.Literal['item_id', 'brand_id', 'category', 'subcategory'] | None = None
) -> None:

    global ITEMS

    agg_dir = Path('datamart/aggs/events')
    output_root = Path('datamart/features/events')
    days = _existing_agg_days(day_from, day_to, agg_dir)
    if not days:
        return

    keys = ['user_id'] + ([item_group] if item_group is not None else [])
    output_dir = output_root / '-'.join(keys)
    output_dir.mkdir(parents=True, exist_ok=True)

    count_columns = {
        ('all_subdomains', action_type): f'num_{action_type}_all_subdomains'
        for action_type in ACTION_TYPES
    } | {
        (subdomain, action_type): f'num_{action_type}_{subdomain}'
        for action_type in ACTION_TYPES
        for subdomain in SUBDOMAINS
    }
    surfaces = ['all_subdomains'] + SUBDOMAINS

    for day in tqdm(days):
        window_days = [d for d in days if day - num_days < d <= day]
        if not window_days:
            continue

        events = pl.concat(
            [pl.scan_parquet(agg_dir / f'{d:05d}.pq') for d in window_days],
            how='vertical',
        )
        rename_map = {
            f'agg_num_{action_type}_all_subdomains': f'num_{action_type}_all_subdomains'
            for action_type in ACTION_TYPES
        } | {
            f'agg_num_{action_type}_{subdomain}': f'num_{action_type}_{subdomain}'
            for action_type in ACTION_TYPES
            for subdomain in SUBDOMAINS
        }
        source_columns = set(events.collect_schema().names())
        events = events.rename({old: new for old, new in rename_map.items() if old in source_columns})

        events = events.join(ITEMS.lazy(), on='item_id', how='left')

        count_aggs = [
            pl.col(column)
            .sum()
            .cast(pl.Float32)
            .alias(_feature_name(f"num_{action_type}{'' if surface == 'all_subdomains' else f'_from_{surface}'}_{num_days}d", keys))
            for surface in surfaces
            for action_type, column in [(action_type, count_columns[(surface, action_type)]) for action_type in ACTION_TYPES]
        ]
        median_aggs = [
            pl.when(pl.col(column) > 0)
            .then(pl.col('price'))
            .otherwise(None)
            .median()
            .cast(pl.Float32)
            .alias(_feature_name(f"median_price_{action_type}{'' if surface == 'all_subdomains' else f'_from_{surface}'}_{num_days}d", keys))
            for surface in surfaces
            for action_type, column in [(action_type, count_columns[(surface, action_type)]) for action_type in ACTION_TYPES]
        ]

        result = events.group_by(keys).agg(count_aggs + median_aggs).collect()
        _write_or_merge_features(result, output_dir / f'{day:05d}.pq', keys)

In [13]:
for item_group in [None, "item_id", "brand_id", "category"]:
    print(f"Calculating features for user_id and {item_group}")
    calculate_user_item_group_features(
        num_days=30, item_group=item_group
    )

Calculating features for user_id and None


100%|██████████| 51/51 [00:07<00:00,  6.61it/s]


Calculating features for user_id and item_id


100%|██████████| 51/51 [00:55<00:00,  1.08s/it]


Calculating features for user_id and brand_id


100%|██████████| 51/51 [00:17<00:00,  2.91it/s]


Calculating features for user_id and category


100%|██████████| 51/51 [00:14<00:00,  3.54it/s]


In [14]:
for item_group in [None, "item_id", "brand_id", "category"]:
    print(f"Calculating features for user_id and {item_group}")
    calculate_user_item_group_features(
        num_days=14, item_group=item_group
    )

Calculating features for user_id and None


100%|██████████| 51/51 [00:04<00:00, 10.26it/s]


Calculating features for user_id and item_id


100%|██████████| 51/51 [00:57<00:00,  1.12s/it]


Calculating features for user_id and brand_id


100%|██████████| 51/51 [00:15<00:00,  3.31it/s]


Calculating features for user_id and category


100%|██████████| 51/51 [00:12<00:00,  4.06it/s]


In [15]:
for item_group in [None, "item_id", "brand_id", "category"]:
    print(f"Calculating features for user_id and {item_group}")
    calculate_user_item_group_features(
        num_days=7, item_group=item_group
    )

Calculating features for user_id and None


100%|██████████| 51/51 [00:03<00:00, 13.68it/s]


Calculating features for user_id and item_id


100%|██████████| 51/51 [00:43<00:00,  1.17it/s]


Calculating features for user_id and brand_id


100%|██████████| 51/51 [00:12<00:00,  3.98it/s]


Calculating features for user_id and category


100%|██████████| 51/51 [00:11<00:00,  4.59it/s]


---

In [16]:
for user_group in [None, "socdem_cluster"]:
    for item_group in ["item_id", "brand_id", "category"]:
        print(f"Calculating features for {user_group} and {item_group}")
        calculate_user_group_item_group_features(
            num_days=30, user_group=user_group, item_group=item_group
        )

Calculating features for None and item_id


100%|██████████| 51/51 [00:08<00:00,  6.16it/s]


Calculating features for None and brand_id


100%|██████████| 51/51 [00:06<00:00,  7.94it/s]


Calculating features for None and category


...

100%|██████████| 51/51 [00:15<00:00,  3.38it/s]


Calculating features for socdem_cluster and brand_id


100%|██████████| 51/51 [00:07<00:00,  6.61it/s]


Calculating features for socdem_cluster and category


100%|██████████| 51/51 [00:08<00:00,  6.10it/s]


In [17]:
for user_group in [None, "socdem_cluster"]:
    for item_group in ["item_id", "brand_id", "category"]:
        print(f"Calculating features for {user_group} and {item_group}")
        calculate_user_group_item_group_features(
            num_days=14, user_group=user_group, item_group=item_group
        )

Calculating features for None and item_id


100%|██████████| 51/51 [00:05<00:00,  8.50it/s]


Calculating features for None and brand_id


100%|██████████| 51/51 [00:04<00:00, 12.55it/s]


Calculating features for None and category


...

100%|██████████| 51/51 [00:13<00:00,  3.87it/s]


Calculating features for socdem_cluster and brand_id


100%|██████████| 51/51 [00:05<00:00,  9.38it/s]


Calculating features for socdem_cluster and category


100%|██████████| 51/51 [00:05<00:00,  8.99it/s]


In [18]:
for user_group in [None, "socdem_cluster"]:
    for item_group in ["item_id", "brand_id", "category"]:
        print(f"Calculating features for {user_group} and {item_group}")
        calculate_user_group_item_group_features(
            num_days=7, user_group=user_group, item_group=item_group
        )

Calculating features for None and item_id


100%|██████████| 51/51 [00:04<00:00, 10.34it/s]


Calculating features for None and brand_id


100%|██████████| 51/51 [00:02<00:00, 18.89it/s]


Calculating features for None and category


...

100%|██████████| 51/51 [00:11<00:00,  4.45it/s]


Calculating features for socdem_cluster and brand_id


100%|██████████| 51/51 [00:03<00:00, 14.18it/s]


Calculating features for socdem_cluster and category


100%|██████████| 51/51 [00:03<00:00, 14.10it/s]


In [ ]:
for path in Path('datamart/features/events').iterdir():
    print(path.name)

brand_id
category
item_id
socdem_cluster-brand_id
socdem_cluster-category
socdem_cluster-item_id
user_id
user_id-brand_id
user_id-category
user_id-item_id


In [20]:
pl.read_parquet('datamart/features/events/user_id/01250.pq').sample(10)

user_id,f_num__user_id__num_view_30d,f_num__user_id__num_click_30d,f_num__user_id__num_clickout_30d,f_num__user_id__num_like_30d,f_num__user_id__num_view_from_u2i_30d,f_num__user_id__num_click_from_u2i_30d,f_num__user_id__num_clickout_from_u2i_30d,f_num__user_id__num_like_from_u2i_30d,f_num__user_id__num_view_from_i2i_30d,f_num__user_id__num_click_from_i2i_30d,f_num__user_id__num_clickout_from_i2i_30d,f_num__user_id__num_like_from_i2i_30d,f_num__user_id__num_view_from_catalog_30d,f_num__user_id__num_click_from_catalog_30d,f_num__user_id__num_clickout_from_catalog_30d,f_num__user_id__num_like_from_catalog_30d,f_num__user_id__num_view_from_search_30d,f_num__user_id__num_click_from_search_30d,f_num__user_id__num_clickout_from_search_30d,f_num__user_id__num_like_from_search_30d,f_num__user_id__num_view_from_other_30d,f_num__user_id__num_click_from_other_30d,f_num__user_id__num_clickout_from_other_30d,f_num__user_id__num_like_from_other_30d,f_num__user_id__median_price_view_30d,f_num__user_id__median_price_click_30d,f_num__user_id__median_price_clickout_30d,f_num__user_id__median_price_like_30d,f_num__user_id__median_price_view_from_u2i_30d,f_num__user_id__median_price_click_from_u2i_30d,f_num__user_id__median_price_clickout_from_u2i_30d,f_num__user_id__median_price_like_from_u2i_30d,f_num__user_id__median_price_view_from_i2i_30d,f_num__user_id__median_price_click_from_i2i_30d,f_num__user_id__median_price_clickout_from_i2i_30d,f_num__user_id__median_price_like_from_i2i_30d,…,f_num__user_id__num_like_from_i2i_7d,f_num__user_id__num_view_from_catalog_7d,f_num__user_id__num_click_from_catalog_7d,f_num__user_id__num_clickout_from_catalog_7d,f_num__user_id__num_like_from_catalog_7d,f_num__user_id__num_view_from_search_7d,f_num__user_id__num_click_from_search_7d,f_num__user_id__num_clickout_from_search_7d,f_num__user_id__num_like_from_search_7d,f_num__user_id__num_view_from_other_7d,f_num__user_id__num_click_from_other_7d,f_num__user_id__num_clickout_from_other_7d,f_num__user_id__num_like_from_other_7d,f_num__user_id__median_price_view_7d,f_num__user_id__median_price_click_7d,f_num__user_id__median_price_clickout_7d,f_num__user_id__median_price_like_7d,f_num__user_id__median_price_view_from_u2i_7d,f_num__user_id__median_price_click_from_u2i_7d,f_num__user_id__median_price_clickout_from_u2i_7d,f_num__user_id__median_price_like_from_u2i_7d,f_num__user_id__median_price_view_from_i2i_7d,f_num__user_id__median_price_click_from_i2i_7d,f_num__user_id__median_price_clickout_from_i2i_7d,f_num__user_id__median_price_like_from_i2i_7d,f_num__user_id__median_price_view_from_catalog_7d,f_num__user_id__median_price_click_from_catalog_7d,f_num__user_id__median_price_clickout_from_catalog_7d,f_num__user_id__median_price_like_from_catalog_7d,f_num__user_id__median_price_view_from_search_7d,f_num__user_id__median_price_click_from_search_7d,f_num__user_id__median_price_clickout_from_search_7d,f_num__user_id__median_price_like_from_search_7d,f_num__user_id__median_price_view_from_other_7d,f_num__user_id__median_price_click_from_other_7d,f_num__user_id__median_price_clickout_from_other_7d,f_num__user_id__median_price_like_from_other_7d
u64,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,…,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32
13153874,8.0,0.0,0.0,0.0,8.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.458067,null,null,null,4.458067,null,null,null,null,null,null,null,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.458067,null,null,null,4.458067,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
41050057,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,-2.114858,null,null,null,null,null,null,null,null,null,null,null,…,0.0,0.0,0.0,0.

In [21]:
pl.read_parquet('datamart/features/events/item_id/01250.pq').sample(10)

item_id,f_num__item_id__median_price_view_30d,f_num__item_id__median_price_click_30d,f_num__item_id__median_price_clickout_30d,f_num__item_id__median_price_like_30d,f_num__item_id__median_price_view_from_u2i_30d,f_num__item_id__median_price_click_from_u2i_30d,f_num__item_id__median_price_clickout_from_u2i_30d,f_num__item_id__median_price_like_from_u2i_30d,f_num__item_id__median_price_view_from_i2i_30d,f_num__item_id__median_price_click_from_i2i_30d,f_num__item_id__median_price_clickout_from_i2i_30d,f_num__item_id__median_price_like_from_i2i_30d,f_num__item_id__median_price_view_from_catalog_30d,f_num__item_id__median_price_click_from_catalog_30d,f_num__item_id__median_price_clickout_from_catalog_30d,f_num__item_id__median_price_like_from_catalog_30d,f_num__item_id__median_price_view_from_search_30d,f_num__item_id__median_price_click_from_search_30d,f_num__item_id__median_price_clickout_from_search_30d,f_num__item_id__median_price_like_from_search_30d,f_num__item_id__median_price_view_from_other_30d,f_num__item_id__median_price_click_from_other_30d,f_num__item_id__median_price_clickout_from_other_30d,f_num__item_id__median_price_like_from_other_30d,f_num__item_id__conv_from_view_to_click_30d,f_num__item_id__conv_from_view_to_click_from_u2i_30d,f_num__item_id__conv_from_view_to_click_from_i2i_30d,f_num__item_id__conv_from_view_to_click_from_catalog_30d,f_num__item_id__conv_from_view_to_click_from_search_30d,f_num__item_id__conv_from_view_to_click_from_other_30d,f_num__item_id__conv_from_view_to_clickout_30d,f_num__item_id__conv_from_view_to_clickout_from_u2i_30d,f_num__item_id__conv_from_view_to_clickout_from_i2i_30d,f_num__item_id__conv_from_view_to_clickout_from_catalog_30d,f_num__item_id__conv_from_view_to_clickout_from_search_30d,f_num__item_id__conv_from_view_to_clickout_from_other_30d,…,f_num__item_id__conv_from_click_to_clickout_from_other_7d,f_num__item_id__conv_from_click_to_like_7d,f_num__item_id__conv_from_click_to_like_from_u2i_7d,f_num__item_id__conv_from_click_to_like_from_i2i_7d,f_num__item_id__conv_from_click_to_like_from_catalog_7d,f_num__item_id__conv_from_click_to_like_from_search_7d,f_num__item_id__conv_from_click_to_like_from_other_7d,f_num__item_id__conv_from_clickout_to_like_7d,f_num__item_id__conv_from_clickout_to_like_from_u2i_7d,f_num__item_id__conv_from_clickout_to_like_from_i2i_7d,f_num__item_id__conv_from_clickout_to_like_from_catalog_7d,f_num__item_id__conv_from_clickout_to_like_from_search_7d,f_num__item_id__conv_from_clickout_to_like_from_other_7d,f_num__item_id__share_view_7d,f_num__item_id__share_view_from_u2i_7d,f_num__item_id__share_view_from_i2i_7d,f_num__item_id__share_view_from_catalog_7d,f_num__item_id__share_view_from_search_7d,f_num__item_id__share_view_from_other_7d,f_num__item_id__share_click_7d,f_num__item_id__share_click_from_u2i_7d,f_num__item_id__share_click_from_i2i_7d,f_num__item_id__share_click_from_catalog_7d,f_num__item_id__share_click_from_search_7d,f_num__item_id__share_click_from_other_7d,f_num__item_id__share_clickout_7d,f_num__item_id__share_clickout_from_u2i_7d,f_num__item_id__share_clickout_from_i2i_7d,f_num__item_id__share_clickout_from_catalog_7d,f_num__item_id__share_clickout_from_search_7d,f_num__item_id__share_clickout_from_other_7d,f_num__item_id__share_like_7d,f_num__item_id__share_like_from_u2i_7d,f_num__item_id__share_like_from_i2i_7d,f_num__item_id__share_like_from_catalog_7d,f_num__item_id__share_like_from_search_7d,f_num__item_id__share_like_from_other_7d
str,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,…,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32
"""nfmcg_26311213""",3.566264,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,3.566264,null,null,null,0.0,null,null,null,null,0.0,0.0,null,null,null,null,0.0,…,null,nu

In [22]:
pl.read_parquet('datamart/features/events/user_id-item_id/01250.pq').sample(10)

user_id,item_id,f_num__user_id_item_id__num_view_30d,f_num__user_id_item_id__num_click_30d,f_num__user_id_item_id__num_clickout_30d,f_num__user_id_item_id__num_like_30d,f_num__user_id_item_id__num_view_from_u2i_30d,f_num__user_id_item_id__num_click_from_u2i_30d,f_num__user_id_item_id__num_clickout_from_u2i_30d,f_num__user_id_item_id__num_like_from_u2i_30d,f_num__user_id_item_id__num_view_from_i2i_30d,f_num__user_id_item_id__num_click_from_i2i_30d,f_num__user_id_item_id__num_clickout_from_i2i_30d,f_num__user_id_item_id__num_like_from_i2i_30d,f_num__user_id_item_id__num_view_from_catalog_30d,f_num__user_id_item_id__num_click_from_catalog_30d,f_num__user_id_item_id__num_clickout_from_catalog_30d,f_num__user_id_item_id__num_like_from_catalog_30d,f_num__user_id_item_id__num_view_from_search_30d,f_num__user_id_item_id__num_click_from_search_30d,f_num__user_id_item_id__num_clickout_from_search_30d,f_num__user_id_item_id__num_like_from_search_30d,f_num__user_id_item_id__num_view_from_other_30d,f_num__user_id_item_id__num_click_from_other_30d,f_num__user_id_item_id__num_clickout_from_other_30d,f_num__user_id_item_id__num_like_from_other_30d,f_num__user_id_item_id__median_price_view_30d,f_num__user_id_item_id__median_price_click_30d,f_num__user_id_item_id__median_price_clickout_30d,f_num__user_id_item_id__median_price_like_30d,f_num__user_id_item_id__median_price_view_from_u2i_30d,f_num__user_id_item_id__median_price_click_from_u2i_30d,f_num__user_id_item_id__median_price_clickout_from_u2i_30d,f_num__user_id_item_id__median_price_like_from_u2i_30d,f_num__user_id_item_id__median_price_view_from_i2i_30d,f_num__user_id_item_id__median_price_click_from_i2i_30d,f_num__user_id_item_id__median_price_clickout_from_i2i_30d,…,f_num__user_id_item_id__num_like_from_i2i_7d,f_num__user_id_item_id__num_view_from_catalog_7d,f_num__user_id_item_id__num_click_from_catalog_7d,f_num__user_id_item_id__num_clickout_from_catalog_7d,f_num__user_id_item_id__num_like_from_catalog_7d,f_num__user_id_item_id__num_view_from_search_7d,f_num__user_id_item_id__num_click_from_search_7d,f_num__user_id_item_id__num_clickout_from_search_7d,f_num__user_id_item_id__num_like_from_search_7d,f_num__user_id_item_id__num_view_from_other_7d,f_num__user_id_item_id__num_click_from_other_7d,f_num__user_id_item_id__num_clickout_from_other_7d,f_num__user_id_item_id__num_like_from_other_7d,f_num__user_id_item_id__median_price_view_7d,f_num__user_id_item_id__median_price_click_7d,f_num__user_id_item_id__median_price_clickout_7d,f_num__user_id_item_id__median_price_like_7d,f_num__user_id_item_id__median_price_view_from_u2i_7d,f_num__user_id_item_id__median_price_click_from_u2i_7d,f_num__user_id_item_id__median_price_clickout_from_u2i_7d,f_num__user_id_item_id__median_price_like_from_u2i_7d,f_num__user_id_item_id__median_price_view_from_i2i_7d,f_num__user_id_item_id__median_price_click_from_i2i_7d,f_num__user_id_item_id__median_price_clickout_from_i2i_7d,f_num__user_id_item_id__median_price_like_from_i2i_7d,f_num__user_id_item_id__median_price_view_from_catalog_7d,f_num__user_id_item_id__median_price_click_from_catalog_7d,f_num__user_id_item_id__median_price_clickout_from_catalog_7d,f_num__user_id_item_id__median_price_like_from_catalog_7d,f_num__user_id_item_id__median_price_view_from_search_7d,f_num__user_id_item_id__median_price_click_from_search_7d,f_num__user_id_item_id__median_price_clickout_from_search_7d,f_num__user_id_item_id__median_price_like_from_search_7d,f_num__user_id_item_id__median_price_view_from_other_7d,f_num__user_id_item_id__median_price_click_from_other_7d,f_num__user_id_item_id__median_price_clickout_from_other_7d,f_num__user_id_item_id__median_price_like_from_other_7d
u64,str,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,…,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32
4918

In [23]:
pl.read_parquet('datamart/features/events/socdem_cluster-brand_id/01250.pq').sample(10)

socdem_cluster,brand_id,f_num__socdem_cluster_brand_id__median_price_view_30d,f_num__socdem_cluster_brand_id__median_price_click_30d,f_num__socdem_cluster_brand_id__median_price_clickout_30d,f_num__socdem_cluster_brand_id__median_price_like_30d,f_num__socdem_cluster_brand_id__median_price_view_from_u2i_30d,f_num__socdem_cluster_brand_id__median_price_click_from_u2i_30d,f_num__socdem_cluster_brand_id__median_price_clickout_from_u2i_30d,f_num__socdem_cluster_brand_id__median_price_like_from_u2i_30d,f_num__socdem_cluster_brand_id__median_price_view_from_i2i_30d,f_num__socdem_cluster_brand_id__median_price_click_from_i2i_30d,f_num__socdem_cluster_brand_id__median_price_clickout_from_i2i_30d,f_num__socdem_cluster_brand_id__median_price_like_from_i2i_30d,f_num__socdem_cluster_brand_id__median_price_view_from_catalog_30d,f_num__socdem_cluster_brand_id__median_price_click_from_catalog_30d,f_num__socdem_cluster_brand_id__median_price_clickout_from_catalog_30d,f_num__socdem_cluster_brand_id__median_price_like_from_catalog_30d,f_num__socdem_cluster_brand_id__median_price_view_from_search_30d,f_num__socdem_cluster_brand_id__median_price_click_from_search_30d,f_num__socdem_cluster_brand_id__median_price_clickout_from_search_30d,f_num__socdem_cluster_brand_id__median_price_like_from_search_30d,f_num__socdem_cluster_brand_id__median_price_view_from_other_30d,f_num__socdem_cluster_brand_id__median_price_click_from_other_30d,f_num__socdem_cluster_brand_id__median_price_clickout_from_other_30d,f_num__socdem_cluster_brand_id__median_price_like_from_other_30d,f_num__socdem_cluster_brand_id__conv_from_view_to_click_30d,f_num__socdem_cluster_brand_id__conv_from_view_to_click_from_u2i_30d,f_num__socdem_cluster_brand_id__conv_from_view_to_click_from_i2i_30d,f_num__socdem_cluster_brand_id__conv_from_view_to_click_from_catalog_30d,f_num__socdem_cluster_brand_id__conv_from_view_to_click_from_search_30d,f_num__socdem_cluster_brand_id__conv_from_view_to_click_from_other_30d,f_num__socdem_cluster_brand_id__conv_from_view_to_clickout_30d,f_num__socdem_cluster_brand_id__conv_from_view_to_clickout_from_u2i_30d,f_num__socdem_cluster_brand_id__conv_from_view_to_clickout_from_i2i_30d,f_num__socdem_cluster_brand_id__conv_from_view_to_clickout_from_catalog_30d,f_num__socdem_cluster_brand_id__conv_from_view_to_clickout_from_search_30d,…,f_num__socdem_cluster_brand_id__conv_from_click_to_clickout_from_other_7d,f_num__socdem_cluster_brand_id__conv_from_click_to_like_7d,f_num__socdem_cluster_brand_id__conv_from_click_to_like_from_u2i_7d,f_num__socdem_cluster_brand_id__conv_from_click_to_like_from_i2i_7d,f_num__socdem_cluster_brand_id__conv_from_click_to_like_from_catalog_7d,f_num__socdem_cluster_brand_id__conv_from_click_to_like_from_search_7d,f_num__socdem_cluster_brand_id__conv_from_click_to_like_from_other_7d,f_num__socdem_cluster_brand_id__conv_from_clickout_to_like_7d,f_num__socdem_cluster_brand_id__conv_from_clickout_to_like_from_u2i_7d,f_num__socdem_cluster_brand_id__conv_from_clickout_to_like_from_i2i_7d,f_num__socdem_cluster_brand_id__conv_from_clickout_to_like_from_catalog_7d,f_num__socdem_cluster_brand_id__conv_from_clickout_to_like_from_search_7d,f_num__socdem_cluster_brand_id__conv_from_clickout_to_like_from_other_7d,f_num__socdem_cluster_brand_id__share_view_7d,f_num__socdem_cluster_brand_id__share_view_from_u2i_7d,f_num__socdem_cluster_brand_id__share_view_from_i2i_7d,f_num__socdem_cluster_brand_id__share_view_from_catalog_7d,f_num__socdem_cluster_brand_id__share_view_from_search_7d,f_num__socdem_cluster_brand_id__share_view_from_other_7d,f_num__socdem_cluster_brand_id__share_click_7d,f_num__socdem_cluster_brand_id__share_click_from_u2i_7d,f_num__socdem_cluster_brand_id__share_click_from_i2i_7d,f_num__socdem_cluster_brand_id__share_click_from_catalog_7d,f_num__socdem_cluster_brand_id__share_click_from_search_7d,f_num__socdem_cluster_brand_id__share_click_from_other_7d,f_num__socdem_cluster_brand_id__share_clickout_7d,f_num__socdem_cluster_brand_id__shar

---

In [24]:
pl.read_parquet('datamart/features/events/user_id/01250.pq').sample(10)

user_id,f_num__user_id__num_view_30d,f_num__user_id__num_click_30d,f_num__user_id__num_clickout_30d,f_num__user_id__num_like_30d,f_num__user_id__num_view_from_u2i_30d,f_num__user_id__num_click_from_u2i_30d,f_num__user_id__num_clickout_from_u2i_30d,f_num__user_id__num_like_from_u2i_30d,f_num__user_id__num_view_from_i2i_30d,f_num__user_id__num_click_from_i2i_30d,f_num__user_id__num_clickout_from_i2i_30d,f_num__user_id__num_like_from_i2i_30d,f_num__user_id__num_view_from_catalog_30d,f_num__user_id__num_click_from_catalog_30d,f_num__user_id__num_clickout_from_catalog_30d,f_num__user_id__num_like_from_catalog_30d,f_num__user_id__num_view_from_search_30d,f_num__user_id__num_click_from_search_30d,f_num__user_id__num_clickout_from_search_30d,f_num__user_id__num_like_from_search_30d,f_num__user_id__num_view_from_other_30d,f_num__user_id__num_click_from_other_30d,f_num__user_id__num_clickout_from_other_30d,f_num__user_id__num_like_from_other_30d,f_num__user_id__median_price_view_30d,f_num__user_id__median_price_click_30d,f_num__user_id__median_price_clickout_30d,f_num__user_id__median_price_like_30d,f_num__user_id__median_price_view_from_u2i_30d,f_num__user_id__median_price_click_from_u2i_30d,f_num__user_id__median_price_clickout_from_u2i_30d,f_num__user_id__median_price_like_from_u2i_30d,f_num__user_id__median_price_view_from_i2i_30d,f_num__user_id__median_price_click_from_i2i_30d,f_num__user_id__median_price_clickout_from_i2i_30d,f_num__user_id__median_price_like_from_i2i_30d,…,f_num__user_id__num_like_from_i2i_7d,f_num__user_id__num_view_from_catalog_7d,f_num__user_id__num_click_from_catalog_7d,f_num__user_id__num_clickout_from_catalog_7d,f_num__user_id__num_like_from_catalog_7d,f_num__user_id__num_view_from_search_7d,f_num__user_id__num_click_from_search_7d,f_num__user_id__num_clickout_from_search_7d,f_num__user_id__num_like_from_search_7d,f_num__user_id__num_view_from_other_7d,f_num__user_id__num_click_from_other_7d,f_num__user_id__num_clickout_from_other_7d,f_num__user_id__num_like_from_other_7d,f_num__user_id__median_price_view_7d,f_num__user_id__median_price_click_7d,f_num__user_id__median_price_clickout_7d,f_num__user_id__median_price_like_7d,f_num__user_id__median_price_view_from_u2i_7d,f_num__user_id__median_price_click_from_u2i_7d,f_num__user_id__median_price_clickout_from_u2i_7d,f_num__user_id__median_price_like_from_u2i_7d,f_num__user_id__median_price_view_from_i2i_7d,f_num__user_id__median_price_click_from_i2i_7d,f_num__user_id__median_price_clickout_from_i2i_7d,f_num__user_id__median_price_like_from_i2i_7d,f_num__user_id__median_price_view_from_catalog_7d,f_num__user_id__median_price_click_from_catalog_7d,f_num__user_id__median_price_clickout_from_catalog_7d,f_num__user_id__median_price_like_from_catalog_7d,f_num__user_id__median_price_view_from_search_7d,f_num__user_id__median_price_click_from_search_7d,f_num__user_id__median_price_clickout_from_search_7d,f_num__user_id__median_price_like_from_search_7d,f_num__user_id__median_price_view_from_other_7d,f_num__user_id__median_price_click_from_other_7d,f_num__user_id__median_price_clickout_from_other_7d,f_num__user_id__median_price_like_from_other_7d
u64,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,…,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32
31369982,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,0.0,0.0,0.0,1.644707,null,null,null,null,null,null,null,null,null,null,null,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,0.0,0.0,0.0,1.644707,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1.644707,null,null,null
54559838,11.0,0.0,0.0,0.0,11.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.353439,null,null,null,2.353439,null,null,null,null,null,null,null,…,0.0,0.0,0.0,0

In [25]:
pl.read_parquet('datamart/features/events/item_id/01250.pq').sample(10)

item_id,f_num__item_id__median_price_view_30d,f_num__item_id__median_price_click_30d,f_num__item_id__median_price_clickout_30d,f_num__item_id__median_price_like_30d,f_num__item_id__median_price_view_from_u2i_30d,f_num__item_id__median_price_click_from_u2i_30d,f_num__item_id__median_price_clickout_from_u2i_30d,f_num__item_id__median_price_like_from_u2i_30d,f_num__item_id__median_price_view_from_i2i_30d,f_num__item_id__median_price_click_from_i2i_30d,f_num__item_id__median_price_clickout_from_i2i_30d,f_num__item_id__median_price_like_from_i2i_30d,f_num__item_id__median_price_view_from_catalog_30d,f_num__item_id__median_price_click_from_catalog_30d,f_num__item_id__median_price_clickout_from_catalog_30d,f_num__item_id__median_price_like_from_catalog_30d,f_num__item_id__median_price_view_from_search_30d,f_num__item_id__median_price_click_from_search_30d,f_num__item_id__median_price_clickout_from_search_30d,f_num__item_id__median_price_like_from_search_30d,f_num__item_id__median_price_view_from_other_30d,f_num__item_id__median_price_click_from_other_30d,f_num__item_id__median_price_clickout_from_other_30d,f_num__item_id__median_price_like_from_other_30d,f_num__item_id__conv_from_view_to_click_30d,f_num__item_id__conv_from_view_to_click_from_u2i_30d,f_num__item_id__conv_from_view_to_click_from_i2i_30d,f_num__item_id__conv_from_view_to_click_from_catalog_30d,f_num__item_id__conv_from_view_to_click_from_search_30d,f_num__item_id__conv_from_view_to_click_from_other_30d,f_num__item_id__conv_from_view_to_clickout_30d,f_num__item_id__conv_from_view_to_clickout_from_u2i_30d,f_num__item_id__conv_from_view_to_clickout_from_i2i_30d,f_num__item_id__conv_from_view_to_clickout_from_catalog_30d,f_num__item_id__conv_from_view_to_clickout_from_search_30d,f_num__item_id__conv_from_view_to_clickout_from_other_30d,…,f_num__item_id__conv_from_click_to_clickout_from_other_7d,f_num__item_id__conv_from_click_to_like_7d,f_num__item_id__conv_from_click_to_like_from_u2i_7d,f_num__item_id__conv_from_click_to_like_from_i2i_7d,f_num__item_id__conv_from_click_to_like_from_catalog_7d,f_num__item_id__conv_from_click_to_like_from_search_7d,f_num__item_id__conv_from_click_to_like_from_other_7d,f_num__item_id__conv_from_clickout_to_like_7d,f_num__item_id__conv_from_clickout_to_like_from_u2i_7d,f_num__item_id__conv_from_clickout_to_like_from_i2i_7d,f_num__item_id__conv_from_clickout_to_like_from_catalog_7d,f_num__item_id__conv_from_clickout_to_like_from_search_7d,f_num__item_id__conv_from_clickout_to_like_from_other_7d,f_num__item_id__share_view_7d,f_num__item_id__share_view_from_u2i_7d,f_num__item_id__share_view_from_i2i_7d,f_num__item_id__share_view_from_catalog_7d,f_num__item_id__share_view_from_search_7d,f_num__item_id__share_view_from_other_7d,f_num__item_id__share_click_7d,f_num__item_id__share_click_from_u2i_7d,f_num__item_id__share_click_from_i2i_7d,f_num__item_id__share_click_from_catalog_7d,f_num__item_id__share_click_from_search_7d,f_num__item_id__share_click_from_other_7d,f_num__item_id__share_clickout_7d,f_num__item_id__share_clickout_from_u2i_7d,f_num__item_id__share_clickout_from_i2i_7d,f_num__item_id__share_clickout_from_catalog_7d,f_num__item_id__share_clickout_from_search_7d,f_num__item_id__share_clickout_from_other_7d,f_num__item_id__share_like_7d,f_num__item_id__share_like_from_u2i_7d,f_num__item_id__share_like_from_i2i_7d,f_num__item_id__share_like_from_catalog_7d,f_num__item_id__share_like_from_search_7d,f_num__item_id__share_like_from_other_7d
str,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,…,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32
"""nfmcg_22048618""",2.421271,null,null,null,null,null,null,null,2.421271,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0.0,null,0.0,null,null,null,0.0,null,0.0,null,null,null,…,null,nu

In [26]:
pl.read_parquet('datamart/features/events/user_id-item_id/01250.pq').sample(10)

user_id,item_id,f_num__user_id_item_id__num_view_30d,f_num__user_id_item_id__num_click_30d,f_num__user_id_item_id__num_clickout_30d,f_num__user_id_item_id__num_like_30d,f_num__user_id_item_id__num_view_from_u2i_30d,f_num__user_id_item_id__num_click_from_u2i_30d,f_num__user_id_item_id__num_clickout_from_u2i_30d,f_num__user_id_item_id__num_like_from_u2i_30d,f_num__user_id_item_id__num_view_from_i2i_30d,f_num__user_id_item_id__num_click_from_i2i_30d,f_num__user_id_item_id__num_clickout_from_i2i_30d,f_num__user_id_item_id__num_like_from_i2i_30d,f_num__user_id_item_id__num_view_from_catalog_30d,f_num__user_id_item_id__num_click_from_catalog_30d,f_num__user_id_item_id__num_clickout_from_catalog_30d,f_num__user_id_item_id__num_like_from_catalog_30d,f_num__user_id_item_id__num_view_from_search_30d,f_num__user_id_item_id__num_click_from_search_30d,f_num__user_id_item_id__num_clickout_from_search_30d,f_num__user_id_item_id__num_like_from_search_30d,f_num__user_id_item_id__num_view_from_other_30d,f_num__user_id_item_id__num_click_from_other_30d,f_num__user_id_item_id__num_clickout_from_other_30d,f_num__user_id_item_id__num_like_from_other_30d,f_num__user_id_item_id__median_price_view_30d,f_num__user_id_item_id__median_price_click_30d,f_num__user_id_item_id__median_price_clickout_30d,f_num__user_id_item_id__median_price_like_30d,f_num__user_id_item_id__median_price_view_from_u2i_30d,f_num__user_id_item_id__median_price_click_from_u2i_30d,f_num__user_id_item_id__median_price_clickout_from_u2i_30d,f_num__user_id_item_id__median_price_like_from_u2i_30d,f_num__user_id_item_id__median_price_view_from_i2i_30d,f_num__user_id_item_id__median_price_click_from_i2i_30d,f_num__user_id_item_id__median_price_clickout_from_i2i_30d,…,f_num__user_id_item_id__num_like_from_i2i_7d,f_num__user_id_item_id__num_view_from_catalog_7d,f_num__user_id_item_id__num_click_from_catalog_7d,f_num__user_id_item_id__num_clickout_from_catalog_7d,f_num__user_id_item_id__num_like_from_catalog_7d,f_num__user_id_item_id__num_view_from_search_7d,f_num__user_id_item_id__num_click_from_search_7d,f_num__user_id_item_id__num_clickout_from_search_7d,f_num__user_id_item_id__num_like_from_search_7d,f_num__user_id_item_id__num_view_from_other_7d,f_num__user_id_item_id__num_click_from_other_7d,f_num__user_id_item_id__num_clickout_from_other_7d,f_num__user_id_item_id__num_like_from_other_7d,f_num__user_id_item_id__median_price_view_7d,f_num__user_id_item_id__median_price_click_7d,f_num__user_id_item_id__median_price_clickout_7d,f_num__user_id_item_id__median_price_like_7d,f_num__user_id_item_id__median_price_view_from_u2i_7d,f_num__user_id_item_id__median_price_click_from_u2i_7d,f_num__user_id_item_id__median_price_clickout_from_u2i_7d,f_num__user_id_item_id__median_price_like_from_u2i_7d,f_num__user_id_item_id__median_price_view_from_i2i_7d,f_num__user_id_item_id__median_price_click_from_i2i_7d,f_num__user_id_item_id__median_price_clickout_from_i2i_7d,f_num__user_id_item_id__median_price_like_from_i2i_7d,f_num__user_id_item_id__median_price_view_from_catalog_7d,f_num__user_id_item_id__median_price_click_from_catalog_7d,f_num__user_id_item_id__median_price_clickout_from_catalog_7d,f_num__user_id_item_id__median_price_like_from_catalog_7d,f_num__user_id_item_id__median_price_view_from_search_7d,f_num__user_id_item_id__median_price_click_from_search_7d,f_num__user_id_item_id__median_price_clickout_from_search_7d,f_num__user_id_item_id__median_price_like_from_search_7d,f_num__user_id_item_id__median_price_view_from_other_7d,f_num__user_id_item_id__median_price_click_from_other_7d,f_num__user_id_item_id__median_price_clickout_from_other_7d,f_num__user_id_item_id__median_price_like_from_other_7d
u64,str,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,…,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32
3015

In [27]:
pl.read_parquet('datamart/features/events/socdem_cluster-brand_id/01250.pq').sample(10)

socdem_cluster,brand_id,f_num__socdem_cluster_brand_id__median_price_view_30d,f_num__socdem_cluster_brand_id__median_price_click_30d,f_num__socdem_cluster_brand_id__median_price_clickout_30d,f_num__socdem_cluster_brand_id__median_price_like_30d,f_num__socdem_cluster_brand_id__median_price_view_from_u2i_30d,f_num__socdem_cluster_brand_id__median_price_click_from_u2i_30d,f_num__socdem_cluster_brand_id__median_price_clickout_from_u2i_30d,f_num__socdem_cluster_brand_id__median_price_like_from_u2i_30d,f_num__socdem_cluster_brand_id__median_price_view_from_i2i_30d,f_num__socdem_cluster_brand_id__median_price_click_from_i2i_30d,f_num__socdem_cluster_brand_id__median_price_clickout_from_i2i_30d,f_num__socdem_cluster_brand_id__median_price_like_from_i2i_30d,f_num__socdem_cluster_brand_id__median_price_view_from_catalog_30d,f_num__socdem_cluster_brand_id__median_price_click_from_catalog_30d,f_num__socdem_cluster_brand_id__median_price_clickout_from_catalog_30d,f_num__socdem_cluster_brand_id__median_price_like_from_catalog_30d,f_num__socdem_cluster_brand_id__median_price_view_from_search_30d,f_num__socdem_cluster_brand_id__median_price_click_from_search_30d,f_num__socdem_cluster_brand_id__median_price_clickout_from_search_30d,f_num__socdem_cluster_brand_id__median_price_like_from_search_30d,f_num__socdem_cluster_brand_id__median_price_view_from_other_30d,f_num__socdem_cluster_brand_id__median_price_click_from_other_30d,f_num__socdem_cluster_brand_id__median_price_clickout_from_other_30d,f_num__socdem_cluster_brand_id__median_price_like_from_other_30d,f_num__socdem_cluster_brand_id__conv_from_view_to_click_30d,f_num__socdem_cluster_brand_id__conv_from_view_to_click_from_u2i_30d,f_num__socdem_cluster_brand_id__conv_from_view_to_click_from_i2i_30d,f_num__socdem_cluster_brand_id__conv_from_view_to_click_from_catalog_30d,f_num__socdem_cluster_brand_id__conv_from_view_to_click_from_search_30d,f_num__socdem_cluster_brand_id__conv_from_view_to_click_from_other_30d,f_num__socdem_cluster_brand_id__conv_from_view_to_clickout_30d,f_num__socdem_cluster_brand_id__conv_from_view_to_clickout_from_u2i_30d,f_num__socdem_cluster_brand_id__conv_from_view_to_clickout_from_i2i_30d,f_num__socdem_cluster_brand_id__conv_from_view_to_clickout_from_catalog_30d,f_num__socdem_cluster_brand_id__conv_from_view_to_clickout_from_search_30d,…,f_num__socdem_cluster_brand_id__conv_from_click_to_clickout_from_other_7d,f_num__socdem_cluster_brand_id__conv_from_click_to_like_7d,f_num__socdem_cluster_brand_id__conv_from_click_to_like_from_u2i_7d,f_num__socdem_cluster_brand_id__conv_from_click_to_like_from_i2i_7d,f_num__socdem_cluster_brand_id__conv_from_click_to_like_from_catalog_7d,f_num__socdem_cluster_brand_id__conv_from_click_to_like_from_search_7d,f_num__socdem_cluster_brand_id__conv_from_click_to_like_from_other_7d,f_num__socdem_cluster_brand_id__conv_from_clickout_to_like_7d,f_num__socdem_cluster_brand_id__conv_from_clickout_to_like_from_u2i_7d,f_num__socdem_cluster_brand_id__conv_from_clickout_to_like_from_i2i_7d,f_num__socdem_cluster_brand_id__conv_from_clickout_to_like_from_catalog_7d,f_num__socdem_cluster_brand_id__conv_from_clickout_to_like_from_search_7d,f_num__socdem_cluster_brand_id__conv_from_clickout_to_like_from_other_7d,f_num__socdem_cluster_brand_id__share_view_7d,f_num__socdem_cluster_brand_id__share_view_from_u2i_7d,f_num__socdem_cluster_brand_id__share_view_from_i2i_7d,f_num__socdem_cluster_brand_id__share_view_from_catalog_7d,f_num__socdem_cluster_brand_id__share_view_from_search_7d,f_num__socdem_cluster_brand_id__share_view_from_other_7d,f_num__socdem_cluster_brand_id__share_click_7d,f_num__socdem_cluster_brand_id__share_click_from_u2i_7d,f_num__socdem_cluster_brand_id__share_click_from_i2i_7d,f_num__socdem_cluster_brand_id__share_click_from_catalog_7d,f_num__socdem_cluster_brand_id__share_click_from_search_7d,f_num__socdem_cluster_brand_id__share_click_from_other_7d,f_num__socdem_cluster_brand_id__share_clickout_7d,f_num__socdem_cluster_brand_id__shar

## II. Сбор датасета. (3 балла)

Буду учить модель ранжировать рекомендации, показанные пользователю в рамках одного дня (хотя можно было бы выбрать и другой промежуток). Подготовку обучающих данных разделю на два этапа: сначала соберу “скелет” или базис, а затем создам датасет, приджойнив фичи к этому базису.

### II.I Сбор датасета -> Сбор базиса (1 из 3 баллов)

Базис буду представлять как (`session_id`, `user_id`, `item_id`, `label`). В качестве `session_id` использую конкатенацию `user_id` и `day`, а `label` задам по `action_type`, учитывая порядок: view < click < clickout < like.

Сессии, которые целиком состоят из view, учитывать не буду, потому что по одинаковым элементам сложно оценивать качество сортировки. Если в рамках одной сессии было несколько взаимодействий с одним айтемом, для `label` возьму максимальное значение.

После получения предсказаний модели можно будет сгруппировать базис по `session_id` и получить структуру ([`item_id`], [`label`], [`score`]). Тогда сортировка айтемов по `label` даст реальный порядок, а сортировка по `score` — модельный. На этой структуре уже удобно считать метрики.

Дальше реализую сбор базиса, сохраняя его по дням. Также добавлю фильтрацию сессий по 99-му перцентилю длины.

In [28]:
def build_basis(
    day_from: int,
    day_to: int,
    filter_99: bool = True,
    output_dir: Path = Path("output/basis/"),
) -> None:
    raw_dir = Path("datamart/raw/events/")
    output_dir.mkdir(parents=True, exist_ok=True)

    action_to_label = {
        'view': 0,
        'click': 1,
        'clickout': 2,
        'like': 3,
    }

    for day in tqdm(range(day_from, day_to + 1)):
        day_name = str(day).zfill(5)
        day_events = []

        for action_type, label in action_to_label.items():
            path = raw_dir / action_type / f'{day_name}.pq'
            if not path.exists():
                continue

            day_events.append(
                pl.scan_parquet(path)
                .select(
                    pl.concat_str(
                        [pl.col('user_id').cast(pl.Utf8), pl.lit(str(day))],
                        separator='_',
                    ).alias('session_id'),
                    pl.col('user_id'),
                    pl.col('item_id'),
                    pl.lit(label).cast(pl.UInt8).alias('label'),
                )
            )

        if not day_events:
            continue

        basis = (
            pl.concat(day_events, how='vertical')
            .group_by('session_id', 'user_id', 'item_id')
            .agg(pl.col('label').max().alias('label'))
        )

        session_stats = (
            basis
            .group_by('session_id')
            .agg(
                pl.len().alias('session_len'),
                pl.col('label').max().alias('max_label'),
            )
            .filter(pl.col('max_label') > 0)
        )

        if filter_99:
            session_len_99 = session_stats.select(
                pl.col('session_len').quantile(0.99)
            ).collect().item()
            if session_len_99 is None:
                continue
            session_stats = session_stats.filter(pl.col('session_len') <= session_len_99)

        result = (
            basis
            .join(session_stats.select('session_id'), on='session_id', how='inner')
            .select('session_id', 'user_id', 'item_id', 'label')
            .collect()
        )

        result.write_parquet(output_dir / f'{day_name}.pq')

In [29]:
build_basis(day_from=min(DAYS), day_to=max(DAYS), filter_99=True)

100%|██████████| 51/51 [00:03<00:00, 14.52it/s]


In [32]:
for col in basis.columns:
    print(f'{col}: {basis[col].n_unique()}')

session_id: 58929
user_id: 18554
item_id: 19994
label: 4


In [33]:
basis['label'].value_counts().sort('label')

label,count
u8,u32
0,1819654
1,147205
2,26243
3,2848


In [34]:
basis.group_by('session_id').agg(pl.len())['len'].describe()

statistic,value
str,f64
"""count""",58929.0
"""null_count""",0.0
"""mean""",33.87042
"""std""",34.702187
"""min""",1.0
"""25%""",10.0
"""50%""",23.0
"""75%""",45.0
"""max""",238.0


### II.II Сбор датасета -> Сбор датасета с фичами (2 из 3 баллов)

Чтобы создать датасет, я приджойню фичи к базису. При этом буду брать фичи за предыдущий день, чтобы избежать ликов: например, если базис построен для 1300-го дня, то фичи для него нужно брать из 1299-го дня.

Помимо числовых признаков добавлю возможность использовать категориальные фичи.

Дальше реализую нужную логику и добавлю возможность читать из файла список фичей, которые необходимо приджойнить к базису.

In [7]:
def join_features(
    df: pl.LazyFrame,
    day: int,
    users: pl.LazyFrame,
    items: pl.LazyFrame,
    features_dir: Path = Path("datamart/features/events/"),
    features_to_use: list[str] | None = None,
) -> pl.LazyFrame:

    feature_day = day - 1
    requested_features = set(features_to_use) if features_to_use is not None else None

    users_small = users.select('user_id', 'socdem_cluster', 'region')
    item_columns = ['item_id', 'brand_id', 'category', 'subcategory', 'price']
    if 'embedding' in items.collect_schema().names():
        item_columns.append('embedding')
    items_small = items.select(item_columns)

    base_features = [
        ('f_cat__socdem_cluster', pl.col('socdem_cluster').cast(pl.Utf8).cast(pl.Categorical)),
        ('f_cat__region', pl.col('region').cast(pl.Utf8).cast(pl.Categorical)),
        ('f_cat__brand_id', pl.col('brand_id').cast(pl.Utf8).cast(pl.Categorical)),
        ('f_cat__category', pl.col('category').cast(pl.Utf8).cast(pl.Categorical)),
        ('f_cat__subcategory', pl.col('subcategory').cast(pl.Utf8).cast(pl.Categorical)),
        ('f_num__price', pl.col('price').cast(pl.Float32)),
    ]
    base_feature_exprs = [
        expr.alias(name)
        for name, expr in base_features
        if requested_features is None or name in requested_features
    ]

    df = (
        df
        .with_columns(pl.lit(day).cast(pl.UInt16).alias('day'))
        .join(users_small, on='user_id', how='left')
        .join(items_small, on='item_id', how='left')
    )
    if base_feature_exprs:
        df = df.with_columns(base_feature_exprs)
    if 'embedding' in item_columns and (requested_features is None or 'embedding' in requested_features):
        df = df.with_columns(pl.col('embedding'))

    available_keys = set(df.collect_schema().names())

    if features_dir.exists():
        for feature_group_dir in sorted(features_dir.iterdir()):
            if not feature_group_dir.is_dir():
                continue

            keys = feature_group_dir.name.split('-')
            if not set(keys).issubset(available_keys):
                continue

            feature_path = feature_group_dir / f'{feature_day:05d}.pq'
            if not feature_path.exists():
                available_feature_days = sorted(
                    int(path.stem)
                    for path in feature_group_dir.glob('*.pq')
                    if path.stem.isdigit()
                )
                previous_feature_days = [d for d in available_feature_days if d <= feature_day]
                if not previous_feature_days:
                    continue

                feature_path = feature_group_dir / f'{previous_feature_days[-1]:05d}.pq'

            features = pl.scan_parquet(feature_path)
            feature_columns = [
                col for col in features.collect_schema().names()
                if col not in keys and col.startswith('f_')
            ]
            if requested_features is not None:
                feature_columns = [col for col in feature_columns if col in requested_features]
            if not feature_columns:
                continue

            df = df.join(
                features.select(keys + feature_columns),
                on=keys,
                how='left',
            )

    result_columns = df.collect_schema().names()
    id_columns = ['session_id', 'user_id', 'item_id', 'label']
    feature_columns = [
        col for col in result_columns
        if col.startswith('f_') or col == 'embedding'
    ]

    return df.select(id_columns + feature_columns)


def build_dataset(
    day_from: int,
    day_to: int,
    basis_dir: Path = Path("output/basis/"),
    output_dir: Path = Path("output/dataset/"),
    features_to_use_filepath: Path | None = None,
    users: pl.LazyFrame | None = None,
    items: pl.LazyFrame | None = None,
) -> None:

    output_dir.mkdir(parents=True, exist_ok=True)

    if users is None:
        users = USERS.lazy()
    if items is None:
        items = ITEMS.lazy()

    features_to_use = None
    if features_to_use_filepath is not None:
        text = features_to_use_filepath.read_text(encoding='utf-8').strip()
        if text:
            try:
                parsed = json.loads(text)
                features_to_use = parsed.get('features', parsed) if isinstance(parsed, dict) else parsed
            except json.JSONDecodeError:
                features_to_use = [line.strip() for line in text.splitlines() if line.strip()]

    for day in tqdm(range(day_from, day_to + 1)):
        day_name = str(day).zfill(5)
        basis_path = basis_dir / f'{day_name}.pq'
        if not basis_path.exists():
            continue

        dataset = join_features(
            df=pl.scan_parquet(basis_path),
            day=day,
            users=users,
            items=items,
            features_to_use=features_to_use,
        )

        dataset.collect().write_parquet(output_dir / f'{day_name}.pq')

In [8]:
dataset_output_dir = Path('output/dataset/')
stale_first_day_path = dataset_output_dir / f'{min(DAYS):05d}.pq'
if stale_first_day_path.exists():
    stale_first_day_path.unlink()

build_dataset(day_from=min(DAYS) + 1, day_to=max(DAYS), output_dir=dataset_output_dir)

100%|██████████| 50/50 [00:23<00:00,  2.17it/s]


In [11]:
pl.read_parquet('output/dataset/01251.pq').sample(10)

session_id,user_id,item_id,label,f_cat__socdem_cluster,f_cat__region,f_cat__brand_id,f_cat__category,f_cat__subcategory,f_num__price,f_num__brand_id__median_price_view_30d,f_num__brand_id__median_price_click_30d,f_num__brand_id__median_price_clickout_30d,f_num__brand_id__median_price_like_30d,f_num__brand_id__median_price_view_from_u2i_30d,f_num__brand_id__median_price_click_from_u2i_30d,f_num__brand_id__median_price_clickout_from_u2i_30d,f_num__brand_id__median_price_like_from_u2i_30d,f_num__brand_id__median_price_view_from_i2i_30d,f_num__brand_id__median_price_click_from_i2i_30d,f_num__brand_id__median_price_clickout_from_i2i_30d,f_num__brand_id__median_price_like_from_i2i_30d,f_num__brand_id__median_price_view_from_catalog_30d,f_num__brand_id__median_price_click_from_catalog_30d,f_num__brand_id__median_price_clickout_from_catalog_30d,f_num__brand_id__median_price_like_from_catalog_30d,f_num__brand_id__median_price_view_from_search_30d,f_num__brand_id__median_price_click_from_search_30d,f_num__brand_id__median_price_clickout_from_search_30d,f_num__brand_id__median_price_like_from_search_30d,f_num__brand_id__median_price_view_from_other_30d,f_num__brand_id__median_price_click_from_other_30d,f_num__brand_id__median_price_clickout_from_other_30d,f_num__brand_id__median_price_like_from_other_30d,f_num__brand_id__conv_from_view_to_click_30d,f_num__brand_id__conv_from_view_to_click_from_u2i_30d,f_num__brand_id__conv_from_view_to_click_from_i2i_30d,…,f_num__user_id_item_id__num_like_from_i2i_7d,f_num__user_id_item_id__num_view_from_catalog_7d,f_num__user_id_item_id__num_click_from_catalog_7d,f_num__user_id_item_id__num_clickout_from_catalog_7d,f_num__user_id_item_id__num_like_from_catalog_7d,f_num__user_id_item_id__num_view_from_search_7d,f_num__user_id_item_id__num_click_from_search_7d,f_num__user_id_item_id__num_clickout_from_search_7d,f_num__user_id_item_id__num_like_from_search_7d,f_num__user_id_item_id__num_view_from_other_7d,f_num__user_id_item_id__num_click_from_other_7d,f_num__user_id_item_id__num_clickout_from_other_7d,f_num__user_id_item_id__num_like_from_other_7d,f_num__user_id_item_id__median_price_view_7d,f_num__user_id_item_id__median_price_click_7d,f_num__user_id_item_id__median_price_clickout_7d,f_num__user_id_item_id__median_price_like_7d,f_num__user_id_item_id__median_price_view_from_u2i_7d,f_num__user_id_item_id__median_price_click_from_u2i_7d,f_num__user_id_item_id__median_price_clickout_from_u2i_7d,f_num__user_id_item_id__median_price_like_from_u2i_7d,f_num__user_id_item_id__median_price_view_from_i2i_7d,f_num__user_id_item_id__median_price_click_from_i2i_7d,f_num__user_id_item_id__median_price_clickout_from_i2i_7d,f_num__user_id_item_id__median_price_like_from_i2i_7d,f_num__user_id_item_id__median_price_view_from_catalog_7d,f_num__user_id_item_id__median_price_click_from_catalog_7d,f_num__user_id_item_id__median_price_clickout_from_catalog_7d,f_num__user_id_item_id__median_price_like_from_catalog_7d,f_num__user_id_item_id__median_price_view_from_search_7d,f_num__user_id_item_id__median_price_click_from_search_7d,f_num__user_id_item_id__median_price_clickout_from_search_7d,f_num__user_id_item_id__median_price_like_from_search_7d,f_num__user_id_item_id__median_price_view_from_other_7d,f_num__user_id_item_id__median_price_click_from_other_7d,f_num__user_id_item_id__median_price_clickout_from_other_7d,f_num__user_id_item_id__median_price_like_from_other_7d
str,u64,str,u8,cat,cat,cat,cat,cat,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,…,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32
"""86398778_1251""",86398778,"""nfmcg_22204691""",0,"""20""","""88""","""117885""","""Electronic Devices and Gadgets""","""Portable Audio Electronics""",3.325357,5.232002,5.157027,4.896489,4.521428,5.157027,5.711161,4.896489,null,4.896489,4.521428,4.369532,null,5.709885,

## III Обучение ранжирования (4 балла)

### III.I. Подготовка выборок для обучения/валидации/теста и реализация метрик (1 из 4 баллов)

Полезно будет также написать функцию, считывающую с диска и возвращающую train, val, train_val, и test части датасета. Диапазон будем задавать через дни.

In [13]:
def read_dataset(
    day_from: int,
    n_train_days: int,
    n_val_days: int,
    n_test_days: int,
    dataset_dir: Path = Path("output/dataset/"),
) -> tuple[pl.LazyFrame, pl.LazyFrame, pl.LazyFrame, pl.LazyFrame]:
    if min(n_train_days, n_val_days, n_test_days) <= 0:
        raise ValueError('n_train_days, n_val_days and n_test_days must be positive')

    def _scan_days(days: list[int]) -> pl.LazyFrame:
        paths = [dataset_dir / f'{day:05d}.pq' for day in days]
        missing_paths = [path for path in paths if not path.exists()]
        if missing_paths:
            missing = ', '.join(str(path) for path in missing_paths[:5])
            raise FileNotFoundError(f'Missing dataset files: {missing}')

        return pl.scan_parquet([str(path) for path in paths])

    train_days = list(range(day_from, day_from + n_train_days))
    val_days = list(range(day_from + n_train_days, day_from + n_train_days + n_val_days))
    test_days = list(range(
        day_from + n_train_days + n_val_days,
        day_from + n_train_days + n_val_days + n_test_days,
    ))

    train_df = _scan_days(train_days)
    val_df = _scan_days(val_days)
    train_val_df = _scan_days(train_days + val_days)
    test_df = _scan_days(test_days)

    return train_df, val_df, train_val_df, test_df

In [14]:
train_df, val_df, train_val_df, test_df = read_dataset(
    day_from=1265,
    n_train_days=14,
    n_val_days=1,
    n_test_days=1,
    dataset_dir=Path('output/dataset/')
)
train_df = train_df.collect()
val_df = val_df.collect()
train_val_df = train_val_df.collect()
test_df = test_df.collect()

In [15]:
display(train_val_df.shape)
display(test_df.shape)

(623514, 2098)

(37568, 2098)

Реализую логику расчета метрик по структуре ([`item_id`], [`label`], [`score`]). Эту структуру можно сформировать как внутри функции расчета метрик, так и заранее отдельно.

В качестве основной метрики обязательно посчитаю NDCG@k. Дополнительно добавлю другие метрики на выбор, например hitrate или recall. Также полезно будет считать метрики в разрезе по `label`: например, сколько айтемов с `label=2` попало в топ-10 рекомендаций.

После этого посчитаю метрики в A/A-сеттинге.

In [16]:
class AtKMetric(ABC):
    def __init__(self, k: int):
        self.k = k

    @property
    @abstractmethod
    def name(self) -> str:
        raise NotImplementedError

    @property
    def full_name(self) -> str:
        return f'{self.name}@{self.k}'

    @abstractmethod
    def __call__(self, *, labels_col: str = 'labels', targets_col: str = 'targets') -> pl.Expr:
        raise NotImplementedError


class NdcgAtK(AtKMetric):
    @property
    def name(self) -> str:
        return 'ndcg'

    def __call__(self, *, labels_col: str = 'labels', targets_col: str = 'targets') -> pl.Expr:
        def _dcg(labels: list[int]) -> float:
            return float(sum(
                (2 ** label - 1) / math.log2(rank + 2)
                for rank, label in enumerate(labels[:self.k])
            ))

        def _ndcg(row: dict[str, list[dict]]) -> float:
            preds = row[labels_col]
            targets = row[targets_col]
            true_labels_by_item = {
                target['item_id']: target['label']
                for target in targets
            }
            pred_labels = [
                true_labels_by_item.get(pred['item_id'], 0)
                for pred in preds[:self.k]
            ]
            ideal_labels = sorted(true_labels_by_item.values(), reverse=True)[:self.k]

            ideal_dcg = _dcg(ideal_labels)
            if ideal_dcg == 0:
                return 0.0
            return _dcg(pred_labels) / ideal_dcg

        return pl.struct(labels_col, targets_col).map_elements(
            _ndcg,
            return_dtype=pl.Float64,
        ).mean().alias(self.full_name)


class HitRateAtK(AtKMetric):
    @property
    def name(self) -> str:
        return 'hitrate'

    def __call__(self, *, labels_col: str = 'labels', targets_col: str = 'targets') -> pl.Expr:
        def _hit_rate(row: dict[str, list[dict]]) -> float:
            preds = row[labels_col]
            targets = row[targets_col]
            relevant_items = {
                target['item_id']
                for target in targets
                if target['label'] > 0
            }
            if not relevant_items:
                return 0.0

            pred_items = {pred['item_id'] for pred in preds[:self.k]}
            return float(len(pred_items & relevant_items) > 0)

        return pl.struct(labels_col, targets_col).map_elements(
            _hit_rate,
            return_dtype=pl.Float64,
        ).mean().alias(self.full_name)


class RecallAtK(AtKMetric):
    @property
    def name(self) -> str:
        return 'recall'

    def __call__(self, *, labels_col: str = 'labels', targets_col: str = 'targets') -> pl.Expr:
        def _recall(row: dict[str, list[dict]]) -> float:
            preds = row[labels_col]
            targets = row[targets_col]
            relevant_items = {
                target['item_id']
                for target in targets
                if target['label'] > 0
            }
            if not relevant_items:
                return 0.0

            pred_items = {pred['item_id'] for pred in preds[:self.k]}
            return len(pred_items & relevant_items) / len(relevant_items)

        return pl.struct(labels_col, targets_col).map_elements(
            _recall,
            return_dtype=pl.Float64,
        ).mean().alias(self.full_name)


def evaluate_ranker(
    df: pl.DataFrame,
    ks: list[int] = [1,  5, 10, 20, 50],
    preds_col: str = 'preds',
    targets_col: str = 'targets',
) -> pd.DataFrame:
    metrics = [
        metric(k)
        for k in ks
        for metric in (NdcgAtK, HitRateAtK, RecallAtK)
    ]

    result = df.select([
        metric(labels_col=preds_col, targets_col=targets_col)
        for metric in metrics
    ])
    return pd.DataFrame(result.to_dicts())

In [17]:
metrics_best = evaluate_ranker(
    test_df.with_columns(score=pl.col('label'))
    .group_by('session_id').agg(
        [
            pl.struct('item_id').sort_by('score', descending=True).alias('preds'),
            pl.struct('item_id', 'label').sort_by('label', descending=True).alias('targets'),
        ]
    )
)
metrics_worst = evaluate_ranker(
    test_df.with_columns(score=pl.col('label'))
    .group_by('session_id').agg(
        [
            pl.struct('item_id').sort_by('score', descending=False).alias('preds'),
            pl.struct('item_id', 'label').sort_by('label', descending=True).alias('targets'),
        ]
    )
)
RESULTS = pd.concat([
    metrics_best.set_axis(['best'], axis='index'),
    metrics_worst.set_axis(['worst'], axis='index'), 
])
RESULTS.style.format(precision=5).background_gradient(cmap='Blues')


,ndcg@1,hitrate@1,recall@1,ndcg@5,hitrate@5,recall@5,ndcg@10,hitrate@10,recall@10,ndcg@20,hitrate@20,recall@20,ndcg@50,hitrate@50,recall@50
best,1.00000,1.00000,0.56509,1.00000,1.00000,0.96632,1.00000,1.00000,0.99526,1.00000,1.00000,0.99990,1.00000,1.00000,1.00000
worst,0.09994,0.10835,0.08682,0.15606,0.22291,0.20925,0.22145,0.40166,0.38701,0.27872,0.58385,0.57031,0.35319,0.86128,0.85319


### III.II. Реализация TopPopular бейзлайна (1 из 4 баллов)

Реализую один или несколько бейзлайнов на своё усмотрение и посчитаю для них метрики.

Буду соблюдать консистентность разбиения:

* обучаюсь на train и оцениваю на val
* обучаюсь на train + val и оцениваю на test

При группировке по сессии для расчета метрик обязательно использую `sample(fraction=1.0, shuffle=True)`. Это нужно, чтобы при одинаковых скорах порядок айтемов не брался автоматически из уже корректно отсортированной последовательности.

In [23]:
def top_popular_score(train: pl.DataFrame, eval_df: pl.DataFrame) -> pl.DataFrame:
    item_popularity = (
        train
        .filter(pl.col('label') > 0)
        .group_by('item_id')
        .agg(
            pl.len().alias('num_positive_interactions'),
            pl.col('label').sum().alias('sum_label'),
        )
        .with_columns(
            (pl.col('sum_label') + pl.col('num_positive_interactions') * 1e-6)
            .cast(pl.Float32)
            .alias('score')
        )
        .select('item_id', 'score')
    )

    return (
        eval_df
        .join(item_popularity, on='item_id', how='left')
        .with_columns(pl.col('score').fill_null(0.0).cast(pl.Float32))
    )


def evaluate_scored_candidates(df: pl.DataFrame) -> pd.DataFrame:
    return evaluate_ranker(
        df
        .sample(fraction=1.0, shuffle=True, seed=42)
        .group_by('session_id')
        .agg([
            pl.struct('item_id').sort_by('score', descending=True).alias('preds'),
            pl.struct('item_id', 'label').sort_by('label', descending=True).alias('targets'),
        ])
    )


baseline_val = evaluate_scored_candidates(top_popular_score(train_df, val_df))
baseline = evaluate_scored_candidates(top_popular_score(train_val_df, test_df))
display('Скоры на валидации')
display(baseline_val.set_axis(['baseline_val'], axis='index'))

RESULTS = pd.concat([
    RESULTS.drop(index=['baseline'], errors='ignore'),
    baseline.set_axis(['baseline'], axis='index'),
])
display('Скоры на тесте')
RESULTS.style.format(precision=5).background_gradient(cmap="Blues")

'Скоры на валидации'

,ndcg@1,hitrate@1,recall@1,ndcg@5,hitrate@5,recall@5,ndcg@10,hitrate@10,recall@10,ndcg@20,hitrate@20,recall@20,ndcg@50,hitrate@50,recall@50
baseline_val,0.247487,0.271108,0.158685,0.366961,0.721636,0.463267,0.451772,0.887863,0.680438,0.504655,0.963061,0.835181,0.542895,0.996702,0.963087


'Скоры на тесте'

,ndcg@1,hitrate@1,recall@1,ndcg@5,hitrate@5,recall@5,ndcg@10,hitrate@10,recall@10,ndcg@20,hitrate@20,recall@20,ndcg@50,hitrate@50,recall@50
best,1.00000,1.00000,0.56509,1.00000,1.00000,0.96632,1.00000,1.00000,0.99526,1.00000,1.00000,0.99990,1.00000,1.00000,1.00000
worst,0.09994,0.10835,0.08682,0.15606,0.22291,0.20925,0.22145,0.40166,0.38701,0.27872,0.58385,0.57031,0.35319,0.86128,0.85319
baseline,0.26761,0.29607,0.17794,0.41077,0.75155,0.51620,0.49202,0.91166,0.72414,0.54080,0.97032,0.86941,0.57098,0.99310,0.97057


In [24]:
test_df['user_id'].n_unique()

1449

В целом работа бейзлайна выглядит корректно, метрики вполне себе лучше значений соответствующих графе worst

Кроме того на тесте получилось качество лучшее чем на на валидации, что радует

Идеального ранжирования как и ожидалось не вышло, но полученные результаты отлично подйодут для сравнения с ранкером

### III.III. Реализация обучения градиентного бустинга (2 из 4 баллов)

Обучу ранкер на train-части датасета. В качестве модели можно использовать CatBoost, LGBM или XGBoost; при этом попробую разные постановки задачи: Ranker, Classifier или Regressor.

На val-части посчитаю метрики и подберу гиперпараметры. Чтобы не заоверфититься под один временной срез, можно взять несколько разных срезов для проверки.

После этого обучу итоговую модель на train + val, замерю качество на test и сравню результат с бейзлайном.

Sanity check: если бейзлайн считается по какой-то фиче из датасета, то эта фича обязательно должна быть среди признаков ранкера. Если ранкер с этой фичей показывает результат хуже бейзлайна, значит с обучением или настройкой ранкера что-то не так.

In [42]:
features = [col for col in train_df.columns if col.startswith('f_')]
categorical_features = [col for col in features if col.startswith('f_cat__')]

In [43]:
display(len(features))

2094

Фичей получилось очень много так что для более стабильного (и самое главное на на 100 часов) обучения я хочу удалить часть фичей, которые оказались наиболее шумными/пустыми/неинформативными

In [ ]:
def feature_sparsity_report(
    df: pl.DataFrame,
    features: list[str],
    categorical_features: list[str],
    thresholds: list[float] = [0.90, 0.95, 0.97, 0.98, 0.99, 0.995],
) -> tuple[pd.DataFrame, pd.DataFrame]:
    numeric_features = [col for col in features if col not in categorical_features]

    numeric_stats = df.select([
        (
            (pl.col(col).is_null() | (pl.col(col).fill_null(0) == 0))
            .mean()
            .alias(col)
        )
        for col in tqdm(numeric_features, desc='Считаю пустоту фичей')
    ])
    zero_null_share = numeric_stats.row(0, named=True) if numeric_features else {}

    rows = []
    for col in tqdm(features, desc='Собираю инфу по фичам'):
        rows.append({
            'feature': col,
            'feature_type': 'cat' if col in categorical_features else 'num',
            'zero_null_share': 0.0 if col in categorical_features else zero_null_share[col],
            'n_unique': df[col].n_unique(),
        })

    report = pd.DataFrame(rows)

    summary = []
    total_features = len(features)
    total_categorical = len(categorical_features)
    total_numeric = len(numeric_features)

    for threshold in tqdm(thresholds, desc='Проверяю пороги'):
        keep_mask = (
            ((report['feature_type'] == 'cat') | (report['zero_null_share'] <= threshold))
            & (report['n_unique'] > 1)
        )
        kept = report[keep_mask]
        kept_numeric = int((kept['feature_type'] == 'num').sum())
        kept_categorical = int((kept['feature_type'] == 'cat').sum())

        summary.append({
            'threshold': threshold,
            'total_features': total_features,
            'total_numeric': total_numeric,
            'total_categorical': total_categorical,
            'kept_features': int(len(kept)),
            'kept_numeric': kept_numeric,
            'kept_categorical': kept_categorical,
            'dropped_features': int(total_features - len(kept)),
            'kept_share': len(kept) / total_features if total_features else 0.0,
        })

    return (
        report.sort_values(['feature_type', 'zero_null_share', 'feature']),
        pd.DataFrame(summary),
    )


feature_report, threshold_summary = feature_sparsity_report(
    train_df,
    features,
    categorical_features,
    thresholds=[0.90, 0.95, 0.97, 0.98, 0.99, 0.995]
)

bad_feature_threshold = 0.99
good_feature_mask = (
    ((feature_report['feature_type'] == 'cat') | (feature_report['zero_null_share'] <= bad_feature_threshold))
    & (feature_report['n_unique'] > 1)
)
filtered_features = feature_report.loc[good_feature_mask, 'feature'].tolist()
filtered_categorical_features = [col for col in categorical_features if col in filtered_features]
filtered_numeric_features = [col for col in filtered_features if col not in filtered_categorical_features]

feature_filter_summary = pd.DataFrame([{
    'bad_feature_threshold': bad_feature_threshold,
    'total_features': len(features),
    'filtered_features': len(filtered_features),
    'filtered_numeric_features': len(filtered_numeric_features),
    'filtered_categorical_features': len(filtered_categorical_features),
    'dropped_features': len(features) - len(filtered_features),
}])

feature_report_dir = Path('output') / 'feature_sparsity'
feature_report_dir.mkdir(parents=True, exist_ok=True)

feature_report_path = feature_report_dir / 'feature_sparsity_report.csv'
threshold_summary_path = feature_report_dir / 'feature_sparsity_threshold_summary.csv'

feature_report.to_csv(feature_report_path, index=False)
threshold_summary.to_csv(threshold_summary_path, index=False)


display(feature_filter_summary)
display(threshold_summary)
display(feature_report.head(100))
display(feature_report.tail(100))

Проверяю пороги: 100%|██████████| 6/6 [00:00<00:00, 1000.11it/s]


,bad_feature_threshold,total_features,filtered_features,filtered_numeric_features,filtered_categorical_features,dropped_features
0,0.99,2094,1406,1401,5,688


,threshold,total_features,total_numeric,total_categorical,kept_features,kept_numeric,kept_categorical,dropped_features,kept_share
0,0.900,2094,2089,5,1199,1194,5,895,0.572588
1,0.950,2094,2089,5,1278,1273,5,816,0.610315
2,0.970,2094,2089,5,1336,1331,5,758,0.638013
3,0.980,2094,2089,5,1354,1349,5,740,0.646609
4,0.990,2094,2089,5,1406,1401,5,688,0.671442
5,0.995,2094,2089,5,1448,1443,5,646,0.691500


,feature,feature_type,zero_null_share,n_unique
2,f_cat__brand_id,cat,0.000000,94
3,f_cat__category,cat,0.000000,21
1,f_cat__region,cat,0.000000,91
0,f_cat__socdem_cluster,cat,0.000000,19
4,f_cat__subcategory,cat,0.000000,126
...,...,...,...,...
241,f_num__brand_id__share_click_from_u2i_7d,num,0.006805,602
199,f_num__brand_id__conv_from_view_to_click_from_...,num,0.006867,754
15,f_num__brand_id__median_price_click_from_i2i_30d,num,0.006971,308
74,f_num__brand_id__share_click_from_i2i_30d,num,0.006971,572


,feature,feature_type,zero_null_share,n_unique
1625,f_num__user_id__num_like_from_i2i_7d,num,1.0,2
1589,f_num__user_id__num_like_from_other_14d,num,1.0,2
1541,f_num__user_id__num_like_from_other_30d,num,1.0,2
1637,f_num__user_id__num_like_from_other_7d,num,1.0,2
1585,f_num__user_id__num_like_from_search_14d,num,1.0,2
...,...,...,...,...
1969,f_num__user_id_item_id__num_like_from_search_30d,num,1.0,2
2065,f_num__user_id_item_id__num_like_from_search_7d,num,1.0,2
2005,f_num__user_id_item_id__num_like_from_u2i_14d,num,1.0,2
1957,f_num__user_id_item_id__num_like_from_u2i_30d,num,1.0,2


Для удобства отчёты сохранены в csv файлы. В целом на их основе я пришёл к тому что качественне будет проводить обучение с порогом 0.99 (фича удаляется если количетво пропусков больше) чтобы не потерять важной информации но и удалить совсем непригодную


In [45]:
def train_evaluate_model(params, train_df, val_df, features, categorical_features):

    model_kind = params.pop('model_kind', 'lgbm_ranker')
    model = fit_boosting_model(
        model_kind=model_kind,
        params=params,
        train_df=train_df,
        features=features,
        categorical_features=categorical_features,
    )
    metrics = evaluate_boosting_model(
        model=model,
        model_kind=model_kind,
        eval_df=val_df,
        features=features,
        categorical_features=categorical_features,
    )
    return float(metrics['ndcg@5'].iloc[0])


from catboost import CatBoostRanker, CatBoostRegressor, Pool


MAX_NUMERIC_FEATURES = None
boosting_features = filtered_features.copy()
boosting_categorical_features = filtered_categorical_features.copy()
boosting_numeric_features = [col for col in boosting_features if col not in boosting_categorical_features]
if MAX_NUMERIC_FEATURES is not None:
    boosting_numeric_features = boosting_numeric_features[:MAX_NUMERIC_FEATURES]
    boosting_features = boosting_categorical_features + boosting_numeric_features

display(pd.DataFrame([{
    'bad_feature_threshold': bad_feature_threshold,
    'boosting_features': len(boosting_features),
    'boosting_numeric_features': len(boosting_numeric_features),
    'boosting_categorical_features': len(boosting_categorical_features),
}]))


def _prepare_boosting_frame(
    df: pl.DataFrame,
    features: list[str],
    categorical_features: list[str],
    *,
    sort_by_session: bool = False,
    for_catboost: bool = False,
) -> pd.DataFrame:
    if sort_by_session:
        df = df.sort('session_id')

    selected_columns = ['session_id', 'label'] + features
    selected_df = df.select(selected_columns)
    pdf = pd.DataFrame({col: selected_df[col].to_numpy() for col in selected_columns})
    for col in categorical_features:
        if col not in pdf.columns:
            continue
        pdf[col] = pdf[col].astype('string').fillna('__missing__')
        if not for_catboost:
            pdf[col] = pdf[col].astype('category')

    return pdf


def _group_sizes(pdf: pd.DataFrame) -> np.ndarray:
    return pdf.groupby('session_id', sort=False).size().to_numpy(dtype=np.int32)


def fit_boosting_model(
    model_kind: str,
    params: dict,
    train_df: pl.DataFrame,
    features: list[str],
    categorical_features: list[str],
):
    params = params.copy()
    is_catboost = model_kind.startswith('catboost')
    is_ranker = model_kind.endswith('ranker')

    train_pdf = _prepare_boosting_frame(
        train_df,
        features,
        categorical_features,
        sort_by_session=is_ranker,
        for_catboost=is_catboost,
    )
    x_train = train_pdf[features]
    y_train = train_pdf['label'].to_numpy()

    if model_kind == 'lgbm_ranker':
        model = lgbm.LGBMRanker(
            objective='lambdarank',
            random_state=42,
            n_jobs=-1,
            verbosity=-1,
            **params,
        )
        model.fit(
            x_train,
            y_train,
            group=_group_sizes(train_pdf),
            categorical_feature=categorical_features,
        )
        return model

    if model_kind == 'lgbm_regressor':
        model = lgbm.LGBMRegressor(
            objective='regression',
            random_state=42,
            n_jobs=-1,
            verbosity=-1,
            **params,
        )
        model.fit(x_train, y_train, categorical_feature=categorical_features)
        return model

    cat_features = [features.index(col) for col in categorical_features if col in features]
    if model_kind == 'catboost_ranker':
        train_pool = Pool(
            x_train,
            y_train,
            group_id=train_pdf['session_id'],
            cat_features=cat_features,
        )
        model = CatBoostRanker(
            loss_function='YetiRank',
            random_seed=42,
            verbose=False,
            allow_writing_files=False,
            **params,
        )
        model.fit(train_pool)
        return model

    if model_kind == 'catboost_regressor':
        train_pool = Pool(x_train, y_train, cat_features=cat_features)
        model = CatBoostRegressor(
            loss_function='RMSE',
            random_seed=42,
            verbose=False,
            allow_writing_files=False,
            **params,
        )
        model.fit(train_pool)
        return model

    raise ValueError(f'Unknown model_kind: {model_kind}')


def score_boosting_model(
    model,
    model_kind: str,
    eval_df: pl.DataFrame,
    features: list[str],
    categorical_features: list[str],
) -> pl.DataFrame:
    is_catboost = model_kind.startswith('catboost')
    eval_pdf = _prepare_boosting_frame(
        eval_df,
        features,
        categorical_features,
        for_catboost=is_catboost,
    )
    x_eval = eval_pdf[features]

    if is_catboost:
        cat_features = [features.index(col) for col in categorical_features if col in features]
        scores = model.predict(Pool(x_eval, cat_features=cat_features))
    else:
        scores = model.predict(x_eval)

    return eval_df.with_columns(pl.Series('score', np.asarray(scores, dtype=np.float32)))


def evaluate_boosting_model(
    model,
    model_kind: str,
    eval_df: pl.DataFrame,
    features: list[str],
    categorical_features: list[str],
) -> pd.DataFrame:
    return evaluate_scored_candidates(
        score_boosting_model(model, model_kind, eval_df, features, categorical_features)
    )


def objective(trial):
    params = {
        'model_kind': 'lgbm_ranker',
        'n_estimators': trial.suggest_int('n_estimators', 80, 180),
        'learning_rate': trial.suggest_float('learning_rate', 0.03, 0.12, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 31, 127),
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 120),
        'subsample': trial.suggest_float('subsample', 0.7, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-2, 10.0, log=True),
    }
    return train_evaluate_model(
        params,
        train_df,
        val_df,
        boosting_features,
        boosting_categorical_features,
    )


study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
print('Подбираю параметры для LGBM ranker')
study.optimize(objective, n_trials=5, show_progress_bar=True)
print(f'\nЛучший ndcg@5: {study.best_value:.5f}')
print('Лучшие параметры:', study.best_params)

,bad_feature_threshold,boosting_features,boosting_numeric_features,boosting_categorical_features
0,0.99,1406,1401,5


[I 2026-04-26 21:43:15,028] A new study created in memory with name: no-name-05e022fc-460f-4e05-a3f8-24b1ffb967cf


Подбираю параметры для LGBM ranker


Best trial: 0. Best value: 0.391274:  20%|██        | 1/5 [01:35<06:20, 95.15s/it]

[I 2026-04-26 21:44:50,162] Trial 0 finished with value: 0.3912742742421349 and parameters: {'n_estimators': 117, 'learning_rate': 0.11207488496994687, 'num_leaves': 102, 'min_child_samples': 80, 'subsample': 0.7468055921327309, 'colsample_bytree': 0.7467983561008608, 'reg_lambda': 0.014936568554617643}. Best is trial 0 with value: 0.3912742742421349.


...

[I 2026-04-26 21:47:40,801] Trial 2 finished with value: 0.40864283201211005 and parameters: {'n_estimators': 98, 'learning_rate': 0.038684926168761874, 'num_leaves': 60, 'min_child_samples': 73, 'subsample': 0.8295835055926347, 'colsample_bytree': 0.7873687420594125, 'reg_lambda': 0.6847920095574779}. Best is trial 2 with value: 0.40864283201211005.


Best trial: 2. Best value: 0.408643:  80%|████████  | 4/5 [05:36<01:19, 79.40s/it]

[I 2026-04-26 21:48:51,489] Trial 3 finished with value: 0.40743995499267716 and parameters: {'n_estimators': 94, 'learning_rate': 0.04497900658389178, 'num_leaves': 66, 'min_child_samples': 66, 'subsample': 0.935552788417904, 'colsample_bytree': 0.7599021346475079, 'reg_lambda': 0.34890188454913873}. Best is trial 2 with value: 0.40864283201211005.


Best trial: 4. Best value: 0.409503: 100%|██████████| 5/5 [07:32<00:00, 90.44s/it]

[I 2026-04-26 21:50:47,214] Trial 4 finished with value: 0.4095026516570856 and parameters: {'n_estimators': 139, 'learning_rate': 0.031995373905230294, 'num_leaves': 89, 'min_child_samples': 37, 'subsample': 0.7195154778955838, 'colsample_bytree': 0.984665661176, 'reg_lambda': 7.886714129990489}. Best is trial 4 with value: 0.4095026516570856.

Лучший ndcg@5: 0.40950


In [47]:
best_lgbm_params = {
    'n_estimators': 139,
    'learning_rate': 0.031995373905230294,
    'num_leaves': 89,
    'min_child_samples': 37,
    'subsample': 0.7195154778955838,
    'colsample_bytree': 0.984665661176,
    'reg_lambda': 7.886714129990489,
}
if 'study' in globals() and len(study.trials) > 0:
    best_lgbm_params = study.best_params.copy()

catboost_depth = int(np.clip(round(np.log2(best_lgbm_params['num_leaves'])), 4, 8))

lgbm_ranker_params = best_lgbm_params.copy()
lgbm_regressor_params = best_lgbm_params.copy()

catboost_ranker_params = {
    'iterations': best_lgbm_params['n_estimators'],
    'learning_rate': best_lgbm_params['learning_rate'],
    'depth': catboost_depth,
    'l2_leaf_reg': best_lgbm_params['reg_lambda'],
}
catboost_regressor_params = catboost_ranker_params.copy()

display(pd.DataFrame([
    {'model': 'lgbm_ranker', **lgbm_ranker_params},
    {'model': 'lgbm_regressor', **lgbm_regressor_params},
    {'model': 'catboost_ranker', **catboost_ranker_params},
    {'model': 'catboost_regressor', **catboost_regressor_params},
]))

model_specs = [
    ('lgbm_ranker', 'lgbm_ranker', lgbm_ranker_params),
    ('lgbm_regressor', 'lgbm_regressor', lgbm_regressor_params),
    ('catboost_ranker', 'catboost_ranker', catboost_ranker_params),
    ('catboost_regressor', 'catboost_regressor', catboost_regressor_params),
]

val_rows = []
test_rows = []
for model_name, model_kind, params in tqdm(model_specs, desc='Обучаю и проверяю модели'):
    val_model = fit_boosting_model(
        model_kind=model_kind,
        params=params,
        train_df=train_df,
        features=boosting_features,
        categorical_features=boosting_categorical_features,
    )
    val_metrics = evaluate_boosting_model(
        model=val_model,
        model_kind=model_kind,
        eval_df=val_df,
        features=boosting_features,
        categorical_features=boosting_categorical_features,
    )
    val_metrics.index = [model_name]
    val_rows.append(val_metrics)

    test_model = fit_boosting_model(
        model_kind=model_kind,
        params=params,
        train_df=train_val_df,
        features=boosting_features,
        categorical_features=boosting_categorical_features,
    )
    test_metrics = evaluate_boosting_model(
        model=test_model,
        model_kind=model_kind,
        eval_df=test_df,
        features=boosting_features,
        categorical_features=boosting_categorical_features,
    )
    test_metrics.index = [model_name]
    test_rows.append(test_metrics)

boosting_val_results = pd.concat(val_rows)
boosting_test_results = pd.concat(test_rows)
display('Скоры бустингов на валидации')
display(boosting_val_results.style.format(precision=5).background_gradient(cmap='Blues'))
display('Скоры бустингов на тесте')
display(boosting_test_results.style.format(precision=5).background_gradient(cmap='Blues'))

best_model_name = boosting_val_results['ndcg@5'].idxmax()
ranker = boosting_test_results.loc[best_model_name].to_dict()
RESULTS = RESULTS.drop(index=['ranker'], errors='ignore')
RESULTS = pd.concat([
    RESULTS,
    pd.DataFrame(ranker, index=['ranker']),
])
RESULTS.style.format(precision=5).background_gradient(cmap='Blues')

,model,n_estimators,learning_rate,num_leaves,min_child_samples,subsample,colsample_bytree,reg_lambda,iterations,depth,l2_leaf_reg
0,lgbm_ranker,139.0,0.031995,89.0,37.0,0.719515,0.984666,7.886714,NaN,NaN,NaN
1,lgbm_regressor,139.0,0.031995,89.0,37.0,0.719515,0.984666,7.886714,NaN,NaN,NaN
2,catboost_ranker,NaN,0.031995,NaN,NaN,NaN,NaN,NaN,139.0,6.0,7.886714
3,catboost_regressor,NaN,0.031995,NaN,NaN,NaN,NaN,NaN,139.0,6.0,7.886714


Обучаю и проверяю модели: 100%|██████████| 4/4 [15:04<00:00, 226.17s/it]


'Скоры бустингов на валидации'

,ndcg@1,hitrate@1,recall@1,ndcg@5,hitrate@5,recall@5,ndcg@10,hitrate@10,recall@10,ndcg@20,hitrate@20,recall@20,ndcg@50,hitrate@50,recall@50
lgbm_ranker,0.28788,0.31596,0.18065,0.40950,0.76979,0.50833,0.48699,0.90831,0.70241,0.53917,0.97098,0.85421,0.57398,0.99604,0.96829
lgbm_regressor,0.28801,0.31069,0.17692,0.40550,0.76517,0.50166,0.48471,0.90633,0.70430,0.53752,0.97427,0.85895,0.57127,0.99802,0.96952
catboost_ranker,0.28860,0.31794,0.18317,0.40720,0.76913,0.50449,0.48842,0.90699,0.70709,0.53732,0.97032,0.85070,0.57203,0.99406,0.96419
catboost_regressor,0.28289,0.30871,0.17333,0.40010,0.75528,0.49363,0.48199,0.89776,0.70316,0.53311,0.97098,0.85480,0.56776,0.99736,0.96756


'Скоры бустингов на тесте'

,ndcg@1,hitrate@1,recall@1,ndcg@5,hitrate@5,recall@5,ndcg@10,hitrate@10,recall@10,ndcg@20,hitrate@20,recall@20,ndcg@50,hitrate@50,recall@50
lgbm_ranker,0.30234,0.33816,0.19768,0.44124,0.79020,0.54951,0.52035,0.92616,0.75023,0.56708,0.97723,0.88636,0.59442,0.99655,0.97868
lgbm_regressor,0.31638,0.34714,0.20109,0.44323,0.78261,0.54477,0.52163,0.91856,0.74476,0.56994,0.97723,0.88560,0.59850,0.99655,0.98055
catboost_ranker,0.30109,0.33126,0.19816,0.43825,0.78123,0.54151,0.51708,0.91787,0.74095,0.56591,0.97861,0.88491,0.59298,0.99448,0.97535
catboost_regressor,0.30632,0.33195,0.19812,0.43451,0.76329,0.53158,0.51584,0.91718,0.74077,0.56397,0.97585,0.88088,0.59258,0.99517,0.97674


,ndcg@1,hitrate@1,recall@1,ndcg@5,hitrate@5,recall@5,ndcg@10,hitrate@10,recall@10,ndcg@20,hitrate@20,recall@20,ndcg@50,hitrate@50,recall@50
best,1.00000,1.00000,0.56509,1.00000,1.00000,0.96632,1.00000,1.00000,0.99526,1.00000,1.00000,0.99990,1.00000,1.00000,1.00000
worst,0.09994,0.10835,0.08682,0.15606,0.22291,0.20925,0.22145,0.40166,0.38701,0.27872,0.58385,0.57031,0.35319,0.86128,0.85319
baseline,0.26761,0.29607,0.17794,0.41077,0.75155,0.51620,0.49202,0.91166,0.72414,0.54080,0.97032,0.86941,0.57098,0.99310,0.97057
ranker,0.30234,0.33816,0.19768,0.44124,0.79020,0.54951,0.52035,0.92616,0.75023,0.56708,0.97723,0.88636,0.59442,0.99655,0.97868


Выводы:

>Как и ожидалось бейзлайн был побит с помощью моделей градиентного бустинга, что логично, ведь топ-поп рекомендации не персонализированные в отличие от бустинга с его учётом признаков айтема, юзера, агрегатных признаков и прочих статистик\
Прирост в качестве не ярко выражен, но чётко прослеживается по метрикам:\
ndcg@5: 0.41077 -> 0.44124\
hitrate@5: 0.75155 -> 0.79020\
recall@5: 0.51620 -> 0.54951\
Касательно разницы между lgbm ранкером и регрессором из проведённого эксперимента -- на тесте она хоть и невелика, но в пользу регрессора (0.44323 против 0.44124), но тем не менее на валидации ранкер показал себя лучше\
CatBoost также смог обогнать бейзлайн, получив на тесте ndcg@5 = 0.43825 (ранкер) и 0.43451 против 0.41077 у бейзлайна (регрессор). При этом CatBoost обучался с параметрами, согласованными с лучшим LGBM-прогоном, без отдельного полноценного тюнинга. В итоге до lgbm он немного не дотянул что как я думаю связано с тем что в датасете основная масса признаков числовая и агрегатная (категориальных фичей всего 5), из-за чего преимущества катбуста на категориальных признаках здесь раскрылись не полностью
